In [ ]:
pip install neurokit2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 25.3 MB/s eta 0:00:00


In [ ]:
import neurokit2 as nk
print("NeuroKit2 version:", nk.__version__)

NeuroKit2 version: 0.2.13


In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
  ПАЙПЛАЙН ОБРАБОТКИ ЭКГ + ФПГ → СКОРОСТЬ РАСПРОСТРАНЕНИЯ ПУЛЬСОВОЙ ВОЛНЫ
=============================================================================
Структура файла:
  §1  Импорты и настройки
  §2  Загрузка и диагностика данных
  §3  Продвинутая предобработка сигналов
  §4  Детекция R-пиков (3 метода + голосование)
  §5  Детекция волн ФПГ (систолические пики и «ноги»)
  §6  Расчёт PTT с физиологической валидацией
  §7  Расчёт СРПВ
  §8  Визуализация

КЛЮЧЕВЫЕ УЛУЧШЕНИЯ vs предыдущих версий:
  ✓ Удаление спайков через IQR + кубическая интерполяция (вместо медианы)
  ✓ Передискретизация ЭКГ до 200 Гц → Pan-Tompkins работает корректно
  ✓ Детекция «ног» ФПГ через систолические пики (метод «долин между пиками»)
      → намного устойчивее касательных и производной на зашумлённом сигнале
  ✓ Три метода детекции R-пиков + голосование большинством → убирает ложные пики
  ✓ Foot-to-foot PTT (chest-onset → arm-onset) вместо R-to-foot
      → не включает PEP (пред-эжекционный период), точнее СРПВ
  ✓ Медианная оценка PWV (устойчива к выбросам PTT)
  ✓ Диагностические графики на каждом шаге
=============================================================================
"""

# ─────────────────────────────────────────────────────────────────────────────
# §1 · ИМПОРТЫ И НАСТРОЙКИ
# ─────────────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.signal import (butter, filtfilt, find_peaks, savgol_filter,
                           medfilt, resample_poly)
from scipy.interpolate import interp1d, CubicSpline
import matplotlib
matplotlib.use('Agg')           # безголовый режим; смените на 'TkAgg'/'Qt5Agg' для интерактива
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Путь к файлу данных ──────────────────────────────────────────
DATA_FILE   = 'data4ch_0_7.xls'   # <— укажите ваш файл
DISTANCE_M  = 0.57                  # расстояние грудь→рука, метры (уточните!) исходно 0.8

# ─────────────────────────────────────────────────────────────────────────────
# §2 · ЗАГРУЗКА И ДИАГНОСТИКА ДАННЫХ
# ─────────────────────────────────────────────────────────────────────────────
def load_and_diagnose(filepath: str) -> tuple:
    """
    Загружает файл, возвращает (time, ppg_chest_raw, ppg_arm_raw, ecg_raw, fs).
    Печатает диагностику: fs, длина записи, пропуски во времени.
    """
    df = pd.read_csv(filepath, header=None)
    print(f"Размер таблицы: {df.shape[0]} строк × {df.shape[1]} столбцов")

    time          = df.iloc[:, 0].values.astype(float)
    ppg_chest_raw = df.iloc[:, 1].values.astype(float)
    ppg_arm_raw   = df.iloc[:, 2].values.astype(float)
    # канал 3 — мусор, пропускаем
    ecg_raw       = df.iloc[:, 4].values.astype(float)

    # ── Анализ временной оси ──────────────────────────────────────
    dt_vals = np.diff(time)
    dt_med  = np.median(dt_vals)
    fs      = 1.0 / dt_med

    jitter_pct = np.std(dt_vals) / dt_med * 100
    gaps       = np.sum(dt_vals > 3 * dt_med)

    print(f"\n── Диагностика ──────────────────────────────────")
    print(f"Частота дискретизации : {fs:.2f} Гц  (dt_median={dt_med*1000:.3f} мс)")
    print(f"Джиттер временной оси : {jitter_pct:.1f} %")
    print(f"Пропусков во времени  : {gaps}")
    print(f"Длина записи          : {time[-1]-time[0]:.1f} с  ({len(time)} отсчётов)")

    # Диапазоны сырых сигналов
    for name, sig in [('ЭКГ', ecg_raw), ('ФПГ грудь', ppg_chest_raw), ('ФПГ рука', ppg_arm_raw)]:
        print(f"  {name:12s}: min={sig.min():.1f}  max={sig.max():.1f}  "
              f"median={np.median(sig):.1f}  std={sig.std():.1f}")

    if fs < 50:
        print("\n⚠️  ВНИМАНИЕ: fs < 50 Гц — ЭКГ будет передискретизирована до 200 Гц")
        print("   Pan-Tompkins и большинство детекторов R-пиков требуют ≥200 Гц")

    return time, ppg_chest_raw, ppg_arm_raw, ecg_raw, float(fs)


# ─────────────────────────────────────────────────────────────────────────────
# §3 · ПРОДВИНУТАЯ ПРЕДОБРАБОТКА
# ─────────────────────────────────────────────────────────────────────────────

# ── 3.1 Удаление спайков (выбросов чтения АЦП) ───────────────────
def remove_spikes_iqr(sig: np.ndarray, fs: float,
                      iqr_factor: float = 5.0,
                      min_spike_width: int = 1,
                      interp_kind: str = 'cubic') -> np.ndarray:
    """
    Удаляет острые спайки (выбросы АЦП/чтения):
      1. Находит выбросы через IQR (в отличие от σ, устойчив к самим выбросам)
      2. Маркирует окрестность выброса (min_spike_width отсчётов с каждой стороны)
      3. Заменяет сплайновой интерполяцией по «чистым» отсчётам

    Параметры
    ---------
    iqr_factor      – множитель IQR для порога (5.0 → очень явные выбросы,
                      снизьте до 3.0 для более агрессивной очистки)
    min_spike_width – полуширина маски вокруг каждого выброса (отсчёты)
    """
    sig = sig.copy()
    Q1, Q3 = np.percentile(sig, 25), np.percentile(sig, 75)
    IQR = Q3 - Q1
    lower = Q1 - iqr_factor * IQR
    upper = Q3 + iqr_factor * IQR

    bad = (sig < lower) | (sig > upper)

    # Расширяем маску на min_spike_width отсчётов
    if min_spike_width > 0:
        from scipy.ndimage import binary_dilation
        struct = np.ones(2 * min_spike_width + 1, dtype=bool)
        bad = binary_dilation(bad, structure=struct)

    n_bad = bad.sum()
    if n_bad == 0:
        return sig
    if n_bad >= len(sig) - 4:
        # слишком много плохих точек — просто медианный фильтр
        return medfilt(sig, kernel_size=5)

    print(f"    Обнаружено {n_bad} выбросов ({n_bad/len(sig)*100:.1f}%) — интерполяция")
    idx = np.arange(len(sig))
    good = ~bad
    f_interp = interp1d(idx[good], sig[good], kind=interp_kind,
                        bounds_error=False, fill_value=(sig[good][0], sig[good][-1]))
    sig[bad] = f_interp(idx[bad])
    return sig


# ── 3.2 Фильтры ───────────────────────────────────────────────────
def butter_bandpass(data, fs, low, high, order=4):
    nyq = 0.5 * fs
    high = min(high, nyq * 0.999)
    low  = max(low,  0.001)
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, data)

def butter_highpass(data, fs, cutoff=0.5, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff/nyq, btype='high')
    return filtfilt(b, a, data)

def butter_lowpass(data, fs, cutoff=15.0, order=4):
    nyq = 0.5 * fs
    cutoff = min(cutoff, nyq * 0.999)
    b, a = butter(order, cutoff/nyq, btype='low')
    return filtfilt(b, a, data)


# ── 3.3 Коррекция базовой линии ФПГ (Smoothed-cubic вместо highpass) ─
def baseline_correct_ppg(sig: np.ndarray, fs: float,
                          window_sec: float = 2.0) -> np.ndarray:
    """
    Оценивает и вычитает медленно дрейфующую базовую линию ФПГ.
    Скользящий медианный фильтр → сплайн → вычитание.
    Сохраняет форму волны ФПГ лучше, чем highpass (0.5 Гц вместо 1 Гц).
    """
    win = int(window_sec * fs)
    if win % 2 == 0: win += 1
    win = max(win, 5)
    baseline = medfilt(sig, kernel_size=min(win, len(sig) // 2 * 2 - 1))
    return sig - baseline


# ── 3.4 Полный конвейер обработки ─────────────────────────────────
def preprocess_ppg(raw: np.ndarray, fs: float,
                   iqr_factor: float = 5.0) -> np.ndarray:
    """
    Цепочка для ФПГ:
      1. Удаление спайков (IQR + интерполяция)
      2. Коррекция базовой линии (медианный фильтр)
      3. Lowpass 12 Гц (сглаживание)
      4. Min-max нормализация в [0, 1]
    """
    sig = remove_spikes_iqr(raw, fs, iqr_factor=iqr_factor, min_spike_width=2)
    sig = baseline_correct_ppg(sig, fs)
    sig = butter_lowpass(sig, fs, cutoff=min(12.0, fs*0.4))
    sig = (sig - sig.min()) / (sig.max() - sig.min() + 1e-9)
    return sig


def preprocess_ecg(raw: np.ndarray, fs: float) -> np.ndarray:
    """
    Цепочка для ЭКГ:
      1. Удаление спайков (IQR)
      2. Полосовая фильтрация 0.5–40 Гц (или до Найквиста)
      3. Z-score нормализация
    """
    sig = remove_spikes_iqr(raw, fs, iqr_factor=6.0, min_spike_width=1)
    highcut = min(8.0, fs*0.4) #min(8.0, fs*0.4) min(40.0, fs * 0.45)
    sig = butter_bandpass(sig, fs, low=0.5, high=highcut)
    sig = (sig - sig.mean()) / (sig.std() + 1e-9)
    return sig


def upsample_signal(sig: np.ndarray, fs_orig: float,
                    fs_target: float = 200.0) -> tuple:
    """
    Передискретизация методом resample_poly (polyphase FIR, без артефактов).
    Возвращает (upsampled_sig, actual_fs_new).
    """
    if fs_orig >= fs_target:
        return sig, fs_orig
    # Находим простую рациональную дробь p/q ≈ fs_target / fs_orig
    ratio = fs_target / fs_orig
    p = round(ratio * 10)
    q = 10
    sig_up = resample_poly(sig, p, q)
    fs_new = fs_orig * p / q
    return sig_up, fs_new


# ─────────────────────────────────────────────────────────────────────────────
# §4 · ДЕТЕКЦИЯ R-ПИКОВ
# ─────────────────────────────────────────────────────────────────────────────

def _auto_invert(ecg: np.ndarray) -> tuple:
    """
    Возвращает (ecg_proc, was_inverted).
    Критерий: если абсолютный максимум > абсолютного минимума — не инвертируем.
    Учитывает инвертированную ЭКГ из 3 электродов без 4-го.
    """
    pos_exc = np.percentile(ecg, 99) - np.median(ecg)
    neg_exc = np.median(ecg) - np.percentile(ecg, 1)
    if neg_exc > pos_exc:
        print("  → ЭКГ инвертирована, применяем переворот")
        return -ecg, True
    return ecg, False


# ── Метод А: Pan-Tompkins (правильная реализация) ─────────────────
def detect_rpeak_pan_tompkins(ecg: np.ndarray, fs: float) -> np.ndarray:
    """
    Классический Pan-Tompkins 1985.
    Требует fs ≥ 150 Гц для надёжной работы.
    Если fs < 200 Гц — передискретизируйте заранее.
    """
    ecg_proc, was_inv = _auto_invert(ecg)

    # 1. Полосовая фильтрация 5–15 Гц
    ecg_bp = butter_bandpass(ecg_proc, fs, low=5, high=min(15, fs*0.45))

    # 2. Производная
    deriv = np.ediff1d(ecg_bp, to_begin=0)

    # 3. Возведение в квадрат
    sq = deriv ** 2

    # 4. Скользящее интегрирование (окно 150 мс)
    win = max(int(0.150 * fs), 2)
    kernel = np.ones(win) / win
    mwi = np.convolve(sq, kernel, mode='same')

    # 5. Адаптивный порог (порог обновляется каждые 200 мс)
    min_dist = int(0.30 * fs) #Минимальное расстояние
    threshold = 0.5 * np.percentile(mwi, 95) #Пороговый множитель
    peaks, props = find_peaks(mwi, height=threshold, distance=min_dist)

    # 6. Уточнение: ищем реальный пик в ±25 мс вокруг найденного
    half = max(1, int(0.03 * fs)) #0.025
    refined = []
    for p in peaks:
        lo, hi = max(0, p-half), min(len(ecg_proc), p+half+1)
        refined.append(lo + np.argmax(ecg_proc[lo:hi]))
    rpeaks = np.unique(np.array(refined, dtype=int))

    # 7. Если был инвертирован — вернуть индексы минимумов в оригинале
    # (но они те же по положению, только знак у значения другой)
    # Фильтрация по физиологическому ЧСС (35–200 уд/мин)
    rpeaks = _filter_by_rr(rpeaks, fs, bpm_min=35, bpm_max=200)
    return rpeaks


# ── Метод Б: Оператор Теагера–Кайзера (TKEO) ─────────────────────
def detect_rpeak_tkeo(ecg: np.ndarray, fs: float) -> np.ndarray:
    """
    TKEO выделяет высокочастотную энергию R-зубцов.
    Устойчив к форме комплекса QRS.
    """
    ecg_proc, _ = _auto_invert(ecg)
    # Bandpass 8–20 Гц для выделения QRS
    ecg_bp = butter_bandpass(ecg_proc, fs, low=8, high=min(20, fs*0.45))

    # TKEO: ψ[n] = x[n]² - x[n-1]·x[n+1]
    tkeo = ecg_bp[1:-1]**2 - ecg_bp[:-2] * ecg_bp[2:]
    tkeo = np.concatenate([[0], tkeo, [0]])
    tkeo = np.clip(tkeo, 0, None)

    # Сглаживание (окно 40 мс)
    win = max(3, int(0.04 * fs) | 1)
    tkeo_sm = savgol_filter(tkeo, window_length=win, polyorder=2)

    threshold = 0.5 * np.percentile(tkeo_sm, 98) #threshold = 0.35 * np.percentile(mwi, 95)
    min_dist  = int(0.4 * fs) # int(0.30 * fs)
    peaks, _  = find_peaks(tkeo_sm, height=threshold, distance=min_dist)

    half = max(1, int(0.025 * fs))
    refined = [max(0, p-half) + np.argmax(ecg_proc[max(0, p-half):min(len(ecg_proc), p+half+1)])
               for p in peaks]
    rpeaks = np.unique(np.array(refined, dtype=int))
    return _filter_by_rr(rpeaks, fs, bpm_min=35, bpm_max=200)


# ── Метод В: NeuroKit2 (HigherLevel API) ──────────────────────────
def detect_rpeak_neurokit(ecg: np.ndarray, fs: float) -> np.ndarray:
    try:
        import neurokit2 as nk
    except ImportError:
        print("  NeuroKit2 не установлен. Выполните: pip install neurokit2")
        return np.array([], dtype=int)

    ecg_proc, _ = _auto_invert(ecg)
    # Масштабируем к [-1, 1]
    ecg_scaled = ecg_proc / (np.max(np.abs(ecg_proc)) + 1e-9)
    try:
        _, info = nk.ecg_peaks(ecg_scaled, sampling_rate=int(fs),
                               method='hamilton2002',
                               correct_artifacts=True)
        rpeaks = info['ECG_R_Peaks']
        # Постфильтрация с более узким диапазоном ЧСС
        return _filter_by_rr(rpeaks, int(fs), bpm_min=35, bpm_max=150)
    except Exception as e:
        print(f"  NeuroKit2: ошибка обработки: {e}")
        return np.array([], dtype=int)


def _filter_by_rr(rpeaks: np.ndarray, fs: float,
                  bpm_min: float = 35, bpm_max: float = 200) -> np.ndarray:
    """Убирает R-пики, формирующие физиологически невозможные RR-интервалы."""
    if len(rpeaks) < 2:
        return rpeaks
    rr = np.diff(rpeaks) / fs  # в секундах
    valid_rr = (60.0 / bpm_max <= rr) & (rr <= 60.0 / bpm_min)

    # Оставляем пик, если хотя бы один из его RR-интервалов валиден
    keep = np.zeros(len(rpeaks), dtype=bool)
    keep[0]  = valid_rr[0]
    keep[-1] = valid_rr[-1]
    for i in range(1, len(rpeaks)-1):
        keep[i] = valid_rr[i-1] or valid_rr[i]
    return rpeaks[keep]


# ── Голосование трёх методов ──────────────────────────────────────
def detect_rpeaks_ensemble(ecg: np.ndarray, fs: float,
                            upsample_to: float = 200.0,
                            tolerance_ms: float = 50.0) -> np.ndarray:
    """
    Запускает три детектора и объединяет их результаты голосованием.
    Пик считается настоящим, если ≥2 из 3 детекторов нашли его
    в пределах ±tolerance_ms.

    Возвращает индексы в ИСХОДНОЙ временной шкале (fs_orig).
    """
    print("\n[§4] Детекция R-пиков")
    # Передискретизация
    ecg_up, fs_up = upsample_signal(ecg, fs, upsample_to)
    print(f"  ЭКГ передискретизирована: {fs:.1f} → {fs_up:.1f} Гц")

    candidates = {}
    for name, fn in [('Pan-Tompkins', detect_rpeak_pan_tompkins),
                     ('TKEO',         detect_rpeak_tkeo),
                     ('NeuroKit2',    detect_rpeak_neurokit)]:
        r = fn(ecg_up, fs_up)
        candidates[name] = r
        print(f"  {name:15s}: найдено {len(r)} R-пиков")

    # Голосование
    tol = int(tolerance_ms / 1000 * fs_up)
    all_peaks_up = np.sort(np.concatenate(list(candidates.values())))
    if len(all_peaks_up) == 0:
        return np.array([], dtype=int)

    voted = []
    used = np.zeros(len(all_peaks_up), dtype=bool)
    for i, p in enumerate(all_peaks_up):
        if used[i]:
            continue
        cluster = all_peaks_up[np.abs(all_peaks_up - p) <= tol]
        votes = sum(
            1 for name, arr in candidates.items()
            if np.any(np.abs(arr - p) <= tol)
        )
        if votes >= 2:
            # центр кластера
            voted.append(int(np.median(cluster)))
        near = np.where(np.abs(all_peaks_up - p) <= tol)[0]
        used[near] = True

    rpeaks_up = np.unique(np.array(voted, dtype=int))
    rpeaks_up = _filter_by_rr(rpeaks_up, fs_up)

    # Переводим обратно в исходную шкалу (простое масштабирование)
    scale = fs / fs_up
    rpeaks_orig = np.round(rpeaks_up * scale).astype(int)
    rpeaks_orig = np.clip(rpeaks_orig, 0, len(ecg)-1)

    # Уточняем положение в исходном сигнале в окне ±20 мс
    ecg_proc, _ = _auto_invert(ecg)
    half = max(1, int(0.020 * fs))
    refined = []
    for p in rpeaks_orig:
        lo, hi = max(0, p-half), min(len(ecg_proc), p+half+1)
        refined.append(lo + np.argmax(ecg_proc[lo:hi]))
    rpeaks_final = np.unique(np.array(refined, dtype=int))
    rpeaks_final = _filter_by_rr(rpeaks_final, fs)

    print(f"  Ансамблевый результат (≥2 голоса): {len(rpeaks_final)} R-пиков")
    # Оценка ЧСС
    if len(rpeaks_final) >= 2:
        rr_mean = np.median(np.diff(rpeaks_final)) / fs
        print(f"  Медиана RR = {rr_mean*1000:.0f} мс  (ЧСС ≈ {60/rr_mean:.0f} уд/мин)")
    return rpeaks_final


# ─────────────────────────────────────────────────────────────────────────────
# §5 · ДЕТЕКЦИЯ ВОЛН ФПГ
# ─────────────────────────────────────────────────────────────────────────────

def detect_ppg_waves(ppg: np.ndarray, fs: float,
                     min_hr_bpm: float = 35, max_hr_bpm: float = 200
                     ) -> dict:
    """
    Надёжная детекция волн ФПГ МЕТОДОМ ДОЛИН МЕЖДУ ПИКАМИ:
      1. Ищем систолические пики (максимумы каждого кардиоцикла)
      2. Между двумя соседними пиками ищем минимум → «нога» (onset, foot)
      3. Диастолический пик (вторичный горб) ищем во второй половине цикла

    Этот метод значительно устойчивее производной/касательных
    на зашумлённом или артефактном сигнале.

    Возвращает словарь с индексами:
      'sys_peaks'  – систолические пики
      'feet'       – диастолические впадины (onsets)
      'dia_peaks'  – диастолические пики (если есть)
    """
    print("\n[§5] Детекция волн ФПГ")
    min_dist = int(60.0 / max_hr_bpm * fs)
    max_dist = int(60.0 / min_hr_bpm * fs)

    # ── Шаг 1: систолические пики ────────────────────────────────
    # Сглаживаем чуть сильнее для надёжной пиковой детекции
    win = max(5, int(0.06 * fs) | 1)
    ppg_sm = savgol_filter(ppg, window_length=win, polyorder=3)

    prominence_thr = 0.08 * (ppg_sm.max() - ppg_sm.min())
    sys_peaks, props = find_peaks(
        ppg_sm,
        distance=min_dist,
        prominence=prominence_thr,
        width=max(2, int(0.03 * fs))    # QRS-ширина
    )
    print(f"  Систолических пиков: {len(sys_peaks)}")

    # ── Шаг 2: «ноги» — минимумы между соседними пиками ──────────
    feet = []
    for i in range(len(sys_peaks) - 1):
        seg = ppg_sm[sys_peaks[i]: sys_peaks[i+1]]
        local_min = sys_peaks[i] + np.argmin(seg)
        feet.append(local_min)

    # «Нога» перед первым пиком
    if sys_peaks[0] > min_dist:
        pre_seg = ppg_sm[:sys_peaks[0]]
        feet.insert(0, np.argmin(pre_seg))

    feet = np.array(feet, dtype=int)
    print(f"  Найдено feet (onsets): {len(feet)}")

    # ── Шаг 3: диастолические пики (необязательно, в середине цикла) ─
    dia_peaks = []
    for i in range(len(feet) - 1):
        # Диастолический горб обычно в 55–85% от длины цикла
        seg_start = feet[i]
        seg_end   = feet[i+1]
        seg_len   = seg_end - seg_start
        d_start   = seg_start + int(0.55 * seg_len)
        d_end     = seg_start + int(0.90 * seg_len)
        if d_end <= d_start:
            continue
        seg = ppg_sm[d_start:d_end]
        local_max_idx = d_start + np.argmax(seg)
        # Принимаем только если это заметный горб (высота > 20% размаха цикла)
        cycle_range = ppg_sm[seg_start:seg_end].max() - ppg_sm[seg_start:seg_end].min()
        if ppg_sm[local_max_idx] - ppg_sm[feet[i]] > 0.15 * cycle_range:
            dia_peaks.append(local_max_idx)

    dia_peaks = np.array(dia_peaks, dtype=int)
    print(f"  Диастолических пиков: {len(dia_peaks)}")

    return {
        'sys_peaks': sys_peaks,
        'feet':      feet,
        'dia_peaks': dia_peaks,
    }


# ─────────────────────────────────────────────────────────────────────────────
# §6 · РАСЧЁТ PTT С ФИЗИОЛОГИЧЕСКОЙ ВАЛИДАЦИЕЙ
# ─────────────────────────────────────────────────────────────────────────────

def compute_ptt(waves_chest: dict, waves_arm: dict, fs: float,
                method: str = 'foot_to_foot',
                ptt_min_ms: float = 20.0,
                ptt_max_ms: float = 500.0) -> dict:
    """
    Вычисляет PTT (Pulse Transit Time) несколькими методами.

    method:
      'foot_to_foot'   – нога грудного ФПГ → нога ручного ФПГ (ЛУЧШИЙ)
                          Не включает PEP; чистая задержка распространения волны.
      'peak_to_peak'   – систолический пик грудного → систолический пик ручного
      'r_to_foot_arm'  – требует rpeaks (передайте как параметр)

    Возвращает dict с:
      'ptt_raw'   – все PTT до фильтрации (секунды)
      'ptt_valid' – PTT после физиологической фильтрации
      'cycle_times' – временные метки (индексы) для каждого PTT
    """
    print(f"\n[§6] Расчёт PTT (метод: {method})")

    if method == 'foot_to_foot':
        ref   = waves_chest['feet']
        probe = waves_arm['feet']
    elif method == 'peak_to_peak':
        ref   = waves_chest['sys_peaks']
        probe = waves_arm['sys_peaks']
    else:
        raise ValueError(f"Неизвестный метод: {method}")

    ptt_min = ptt_min_ms / 1000
    ptt_max = ptt_max_ms / 1000

    ptt_raw    = []
    cycle_idxs = []

    for r in ref:
        # Ищем ближайший probe после r, но не раньше ptt_min
        candidates = probe[(probe >= r + int(ptt_min * fs)) &
                           (probe <= r + int(ptt_max * fs))]
        if len(candidates) == 0:
            continue
        # Берём первый (ближайший)
        nearest = candidates[0]
        ptt = (nearest - r) / fs
        ptt_raw.append(ptt)
        cycle_idxs.append(r)

    ptt_raw = np.array(ptt_raw)
    cycle_idxs = np.array(cycle_idxs, dtype=int)

    if len(ptt_raw) == 0:
        print("  ⚠️  PTT не найдены — проверьте качество сигнала и параметры")
        return {'ptt_raw': np.array([]), 'ptt_valid': np.array([]), 'cycle_times': np.array([])}

    # Фильтрация выбросов PTT (MAD-метод, устойчив к выбросам)
    median_ptt = np.median(ptt_raw)
    mad = np.median(np.abs(ptt_raw - median_ptt))
    z_scores = np.abs(ptt_raw - median_ptt) / (mad * 1.4826 + 1e-9)
    valid = z_scores < 3.5

    ptt_valid = ptt_raw[valid]
    cycle_valid = cycle_idxs[valid]

    print(f"  Всего пар: {len(ptt_raw)},  после MAD-фильтрации: {len(ptt_valid)}")
    if len(ptt_valid):
        print(f"  PTT = {np.median(ptt_valid)*1000:.1f} ± {np.std(ptt_valid)*1000:.1f} мс  "
              f"(медиана ± std)")

    return {
        'ptt_raw':    ptt_raw,
        'ptt_valid':  ptt_valid,
        'cycle_times': cycle_valid,
    }


# ─────────────────────────────────────────────────────────────────────────────
# §7 · РАСЧЁТ СКОРОСТИ РАСПРОСТРАНЕНИЯ ПУЛЬСОВОЙ ВОЛНЫ
# ─────────────────────────────────────────────────────────────────────────────

def compute_pwv(ptt_result: dict, distance_m: float) -> dict:
    """
    СРПВ = расстояние / PTT
    Использует медиану PTT для робастности.

    Нормы СРПВ:
      Молодые здоровые : 5–8 м/с
      Средний возраст  : 8–12 м/с
      Пожилые/гипертензия: >12 м/с

    Расстояние: от грудного датчика (approx. место аорты) до ладонного датчика.
    """
    ptt_valid = ptt_result['ptt_valid']
    if len(ptt_valid) < 3:
        print("⚠️  Недостаточно валидных PTT для надёжной СРПВ")
        return {}

    ptt_med   = np.median(ptt_valid)
    ptt_std   = np.std(ptt_valid)
    pwv_med   = distance_m / ptt_med
    pwv_std   = distance_m * ptt_std / ptt_med**2

    print(f"\n[§7] СКОРОСТЬ РАСПРОСТРАНЕНИЯ ПУЛЬСОВОЙ ВОЛНЫ")
    print(f"  Расстояние         : {distance_m:.2f} м")
    print(f"  PTT (медиана)      : {ptt_med*1000:.1f} мс")
    print(f"  PTT (std)          : {ptt_std*1000:.1f} мс")
    print(f"  СРПВ               : {pwv_med:.2f} ± {pwv_std:.2f} м/с")
    print(f"  Диапазон PTT       : {ptt_valid.min()*1000:.0f}–{ptt_valid.max()*1000:.0f} мс")

    return {
        'pwv_mps':   pwv_med,
        'pwv_std':   pwv_std,
        'ptt_ms':    ptt_med * 1000,
        'n_cycles':  len(ptt_valid),
    }


# ─────────────────────────────────────────────────────────────────────────────
# §8 · ВИЗУАЛИЗАЦИЯ
# ─────────────────────────────────────────────────────────────────────────────

def plot_overview(time, ecg, ppg_chest, ppg_arm,
                  rpeaks, waves_chest, waves_arm, ptt_result,
                  save_path: str = None, show_sec: float = 20.0):
    """Сводный диагностический график (4 панели)."""

    fig = plt.figure(figsize=(16, 10))
    gs  = gridspec.GridSpec(4, 2, figure=fig, hspace=0.45, wspace=0.3)

    ax_ecg  = fig.add_subplot(gs[0, :])
    ax_ch   = fig.add_subplot(gs[1, :])
    ax_arm  = fig.add_subplot(gs[2, :])
    ax_ptt  = fig.add_subplot(gs[3, 0])
    ax_hist = fig.add_subplot(gs[3, 1])

    mask = time <= (time[0] + show_sec)

    # ── ЭКГ + R-пики ──
    ax_ecg.plot(time[mask], ecg[mask], 'b', lw=0.7, label='ЭКГ (фильтр.)')
    r_in = rpeaks[(rpeaks < mask.sum())]
    ax_ecg.plot(time[r_in], ecg[r_in], 'rv', ms=6, label=f'R-пики (n={len(rpeaks)})')
    ax_ecg.set_ylabel('z-score')
    ax_ecg.set_title('ЭКГ с детектированными R-пиками')
    ax_ecg.legend(loc='upper right', fontsize=8)
    ax_ecg.grid(True, alpha=0.3)

    # ── ФПГ грудь ──
    ax_ch.plot(time[mask], ppg_chest[mask], 'g', lw=0.8, label='ФПГ грудь')
    fc_in = waves_chest['feet'][waves_chest['feet'] < mask.sum()]
    sp_in = waves_chest['sys_peaks'][waves_chest['sys_peaks'] < mask.sum()]
    ax_ch.plot(time[fc_in], ppg_chest[fc_in], 'go', ms=5, label='Feet (onset)')
    ax_ch.plot(time[sp_in], ppg_chest[sp_in], 'g^', ms=5, label='Sys. peak')
    ax_ch.set_ylabel('норм.')
    ax_ch.set_title('ФПГ грудь — детектированные точки')
    ax_ch.legend(loc='upper right', fontsize=8)
    ax_ch.grid(True, alpha=0.3)

    # ── ФПГ рука ──
    ax_arm.plot(time[mask], ppg_arm[mask], color='purple', lw=0.8, label='ФПГ рука')
    fa_in = waves_arm['feet'][waves_arm['feet'] < mask.sum()]
    sa_in = waves_arm['sys_peaks'][waves_arm['sys_peaks'] < mask.sum()]
    ax_arm.plot(time[fa_in], ppg_arm[fa_in], 'mo', ms=5, label='Feet (onset)')
    ax_arm.plot(time[sa_in], ppg_arm[sa_in], 'm^', ms=5, label='Sys. peak')
    ax_arm.set_ylabel('норм.')
    ax_arm.set_xlabel('Время, с')
    ax_arm.set_title('ФПГ рука — детектированные точки')
    ax_arm.legend(loc='upper right', fontsize=8)
    ax_arm.grid(True, alpha=0.3)

    # ── Динамика PTT ──
    ptt_valid = ptt_result['ptt_valid']
    ct = ptt_result['cycle_times']
    if len(ptt_valid):
        ax_ptt.plot(time[ct], ptt_valid * 1000, 'ko-', ms=4, lw=1)
        ax_ptt.axhline(np.median(ptt_valid) * 1000, color='r', ls='--',
                       label=f'Медиана={np.median(ptt_valid)*1000:.0f} мс')
        ax_ptt.set_xlabel('Время, с')
        ax_ptt.set_ylabel('PTT, мс')
        ax_ptt.set_title('Динамика PTT по кардиоциклам')
        ax_ptt.legend(fontsize=8)
        ax_ptt.grid(True, alpha=0.3)

        ax_hist.hist(ptt_valid * 1000, bins=min(20, len(ptt_valid)),
                     color='steelblue', edgecolor='white', alpha=0.8)
        ax_hist.set_xlabel('PTT, мс')
        ax_hist.set_ylabel('Количество')
        ax_hist.set_title('Гистограмма PTT')
        ax_hist.grid(True, alpha=0.3)

    plt.suptitle('Комплексный анализ ЭКГ+ФПГ → СРПВ', fontsize=13, fontweight='bold')

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  График сохранён: {save_path}")
    plt.close()


def plot_single_cycles(ppg_chest, ppg_arm, waves_chest, waves_arm,
                       rpeaks, fs, n_cycles: int = 6, save_path: str = None):
    """Постановка грудного и ручного ФПГ для первых n_cycles циклов."""
    n = min(n_cycles, len(rpeaks) - 1, len(waves_chest['sys_peaks']) - 1)
    if n < 1:
        return
    fig, axes = plt.subplots(n, 1, figsize=(12, 2.5*n), sharex=False)
    if n == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        if i >= len(waves_chest['feet']) - 1:
            break
        # Берём сегмент по feet
        s = waves_chest['feet'][i]
        e = waves_chest['feet'][i+1] if i+1 < len(waves_chest['feet']) else s + int(fs)
        t = np.arange(e - s) / fs * 1000   # мс

        ax.plot(t, ppg_chest[s:e], 'g-', lw=1.2, label='ФПГ грудь')
        ax.plot(t, ppg_arm[s:e],   'm-', lw=1.2, label='ФПГ рука', alpha=0.85)

        # Метки
        ax.axvline(0, color='g', ls=':', lw=1)

        # Нога руки в этом сегменте
        arm_f_in_seg = waves_arm['feet'][(waves_arm['feet'] >= s) & (waves_arm['feet'] < e)]
        for af in arm_f_in_seg:
            ax.axvline((af - s) / fs * 1000, color='m', ls=':', lw=1)
            ptt_ms = (af - s) / fs * 1000
            ax.text((af - s) / fs * 1000 + 2, 0.05, f'PTT≈{ptt_ms:.0f}мс',
                    color='m', fontsize=7, va='bottom')

        ax.set_ylabel('норм.')
        ax.set_title(f'Цикл {i+1}')
        ax.legend(loc='upper right', fontsize=7)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('Время от начала цикла, мс')
    plt.suptitle('Отдельные кардиоциклы: ФПГ грудь vs ФПГ рука', fontsize=12)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  График сохранён: {save_path}")
    plt.close()


# ─────────────────────────────────────────────────────────────────────────────
# §9 · ГЛАВНАЯ ФУНКЦИЯ
# ─────────────────────────────────────────────────────────────────────────────

def run_pipeline(data_file: str = DATA_FILE,
                 distance_m: float = DISTANCE_M,
                 output_dir: str = '.',
                 ptt_method: str = 'foot_to_foot',
                 upsample_ecg_to: float = 200.0) -> dict:
    """
    Запускает полный пайплайн и возвращает словарь результатов.
    """
    import os
    os.makedirs(output_dir, exist_ok=True)

    # §2 Загрузка
    time, ppg_chest_raw, ppg_arm_raw, ecg_raw, fs = load_and_diagnose(data_file)

    # §3 Предобработка
    print("\n[§3] Предобработка сигналов")
    print("  ФПГ грудь...")
    ppg_chest = preprocess_ppg(ppg_chest_raw, fs)
    print("  ФПГ рука...")
    ppg_arm   = preprocess_ppg(ppg_arm_raw,   fs)
    print("  ЭКГ...")
    ecg       = preprocess_ecg(ecg_raw, fs)


    import matplotlib.pyplot as plt
    plt.figure(figsize=(12,3))
    plt.plot(time[:int(20*fs)], ecg[:int(20*fs)])
    plt.title('Предобработанный ЭКГ (первые 20 с)')
    plt.xlabel('Время, с')
    plt.grid(True)
    plt.show()





    # §4 R-пики
    rpeaks = detect_rpeaks_ensemble(ecg, fs, upsample_to=upsample_ecg_to)

    # §5 Волны ФПГ
    waves_chest = detect_ppg_waves(ppg_chest, fs)
    waves_arm   = detect_ppg_waves(ppg_arm,   fs)

    # §6 PTT
    ptt_result = compute_ptt(waves_chest, waves_arm, fs, method=ptt_method)

    # §7 СРПВ
    pwv_result = compute_pwv(ptt_result, distance_m)

    # §8 Визуализация
    print("\n[§8] Построение графиков")
    plot_overview(
        time, ecg, ppg_chest, ppg_arm,
        rpeaks, waves_chest, waves_arm, ptt_result,
        save_path=os.path.join(output_dir, 'overview.png')
    )
    plot_single_cycles(
        ppg_chest, ppg_arm, waves_chest, waves_arm, rpeaks, fs,
        save_path=os.path.join(output_dir, 'cycles.png')
    )

    return {
        'fs': fs, 'rpeaks': rpeaks,
        'waves_chest': waves_chest, 'waves_arm': waves_arm,
        'ptt': ptt_result, 'pwv': pwv_result,
        'signals': {'time': time, 'ecg': ecg,
                    'ppg_chest': ppg_chest, 'ppg_arm': ppg_arm}
    }


# ─────────────────────────────────────────────────────────────────────────────
# ТОЧКА ВХОДА
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    results = run_pipeline(
        data_file   = DATA_FILE,
        distance_m  = DISTANCE_M,
        output_dir  = 'pwv_output',
        ptt_method  = 'foot_to_foot',    # или 'peak_to_peak'
        upsample_ecg_to = 200.0          # Hz для ЭКГ
    )
    print("\n✓ Пайплайн завершён")


Размер таблицы: 32939 строк × 5 столбцов

── Диагностика ──────────────────────────────────
Частота дискретизации : 25.00 Гц  (dt_median=40.000 мс)
Джиттер временной оси : 0.3 %
Пропусков во времени  : 0
Длина записи          : 1317.5 с  (32939 отсчётов)
  ЭКГ         : min=8752171.0  max=8809114.0  median=8789967.0  std=8690.7
  ФПГ грудь   : min=0.0  max=3034814.0  median=2627656.0  std=227235.3
  ФПГ рука    : min=0.0  max=9910248.0  median=9377881.0  std=445568.3

⚠️  ВНИМАНИЕ: fs < 50 Гц — ЭКГ будет передискретизирована до 200 Гц
   Pan-Tompkins и большинство детекторов R-пиков требуют ≥200 Гц

[§3] Предобработка сигналов
  ФПГ грудь...
    Обнаружено 510 выбросов (1.5%) — интерполяция
  ФПГ рука...
    Обнаружено 340 выбросов (1.0%) — интерполяция
  ЭКГ...

[§4] Детекция R-пиков
  ЭКГ передискретизирована: 25.0 → 200.0 Гц
  → ЭКГ инвертирована, применяем переворот
  Pan-Tompkins   : найдено 2317 R-пиков
  → ЭКГ инвертирована, применяем переворот
  TKEO           : найдено 212 R-п



---



---



In [ ]:
# -*- coding: utf-8 -*-
"""
Диагностика R‑пиков с интерактивной визуализацией (Plotly) и подбором порога.
"""
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks, savgol_filter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ─── НАСТРОЙКИ ─────────────────────────
DATA_FILE = 'data4ch_1_1.xls'
THRESHOLD_FACTORS = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40]  # для подбора
MIN_RR_SEC = 0.4          # минимальный RR (сек)
PERCENTILE = 98           # перцентиль энергии TKEO для порога

# ─── ЗАГРУЗКА ──────────────────────────
df = pd.read_csv(DATA_FILE, header=None)
time = df.iloc[:, 0].values.astype(float)
ecg_raw = df.iloc[:, 4].values.astype(float)
fs = 1.0 / np.median(np.diff(time))
print(f"Частота дискретизации: {fs:.2f} Гц")
print(f"Длина записи: {time[-1]-time[0]:.1f} с ({len(ecg_raw)} отсчётов)")

# ─── ФИЛЬТРЫ И ДЕТЕКТОР ────────────────
def bandpass_filter(data, fs, low=0.5, high=40.0, order=4):
    nyq = 0.5 * fs
    low, high = low/nyq, min(high/nyq, 0.99)
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data)

def preprocess_ecg_safe(raw, fs):
    ecg = bandpass_filter(raw, fs, 0.5, min(40.0, fs*0.49))
    return (ecg - ecg.mean()) / (ecg.std() + 1e-8)

def detect_rpeaks_robust(ecg, fs, min_rr_sec=0.5, tkeo_factor=0.3):
    """
    Двухступенчатый детектор:
    1) TKEO -> кандидаты.
    2) Автоматический фильтр по высоте пика (половина медианы высот всех кандидатов).
    """
    # ── Шаг 1: TKEO (чувствительный) ──────────────────────
    pos = np.max(ecg) - np.median(ecg)
    neg = np.median(ecg) - np.min(ecg)
    inverted = neg > pos
    ecg_proc = -ecg if inverted else ecg

    tkeo = ecg_proc[1:-1]**2 - ecg_proc[:-2] * ecg_proc[2:]
    tkeo = np.insert(tkeo, 0, 0)
    win = max(3, int(0.05 * fs) | 1)
    tkeo_smooth = savgol_filter(tkeo, window_length=win, polyorder=2)
    thresh_tkeo = tkeo_factor * np.percentile(tkeo_smooth, 98)
    min_dist = int(min_rr_sec * fs)
    candidates, _ = find_peaks(tkeo_smooth, height=thresh_tkeo, distance=min_dist)

    # ── Шаг 2: уточнение положения и вычисление высоты ────
    half_win = max(1, int(0.03 * fs))
    peak_indices = []
    peak_heights = []
    for p in candidates:
        lo = max(0, p - half_win)
        hi = min(len(ecg), p + half_win + 1)
        local_peak = lo + (np.argmin(ecg[lo:hi]) if inverted else np.argmax(ecg[lo:hi]))
        # Оцениваем базовый уровень как медиану в окне ±1 с вокруг пика
        win_bl = int(1.0 * fs)
        bl_start = max(0, local_peak - win_bl)
        bl_end = min(len(ecg), local_peak + win_bl)
        baseline = np.median(ecg[bl_start:bl_end])
        height = np.abs(ecg[local_peak] - baseline)
        peak_indices.append(local_peak)
        peak_heights.append(height)

    peak_indices = np.array(peak_indices)
    peak_heights = np.array(peak_heights)

    if len(peak_heights) < 2:
        return peak_indices

    # ── Шаг 3: автоматический порог по высоте ─────────────
    # Истинные R‑зубцы дают высоты, близкие друг к другу и значительно выше шума.
    # Порог = 0.5 × медианная высота (можно увеличить до 0.6 при остаточных ложных пиках)
    height_threshold = 0.5 * np.median(peak_heights)
    keep = peak_heights >= height_threshold
    refined = peak_indices[keep]
    return np.unique(refined)

# ─── ПРЕДОБРАБОТКА ─────────────────────
ecg = preprocess_ecg_safe(ecg_raw, fs)

# Подбор только tkeo_factor (амплитудный фильтр теперь автоматический)
print("\nПодбор tkeo_factor (авто‑фильтр высоты):")
for tkeo_f in [0.2, 0.25, 0.3, 0.35, 0.4, 0.5, 0.6, 0.7, 0.8]:
    r = detect_rpeaks_robust(ecg, fs, tkeo_factor=tkeo_f, min_rr_sec=0.4)
    hr = 60.0/(np.median(np.diff(r))/fs) if len(r)>=2 else 0
    print(f"  tkeo_factor={tkeo_f:.2f}  →  пиков: {len(r):4d}  ЧСС≈{hr:.0f} уд/мин")

# Визуализация с выбранным tkeo_factor
BEST_TKEO = 0.8
rpeaks = detect_rpeaks_robust(ecg, fs, tkeo_factor=BEST_TKEO)
print(f"\nВизуализация: tkeo_factor={BEST_TKEO}, найдено {len(rpeaks)} R‑пиков")

# Интерактивный график
fig = make_subplots(rows=1, cols=1, shared_xaxes=True,
                    subplot_titles=('ЭКГ с R‑пиками',))

fig.add_trace(go.Scattergl(x=time, y=ecg, name='ЭКГ', line=dict(color='blue', width=0.8)))
fig.add_trace(go.Scattergl(x=time[rpeaks], y=ecg[rpeaks], mode='markers',
                           name='R‑пики', marker=dict(color='red', size=6, symbol='x')))
fig.update_xaxes(title_text='Время, с', rangeslider=dict(visible=True))
fig.update_yaxes(title_text='Z‑score')
fig.update_layout(height=500, title=f'Детекция R‑пиков (найдено {len(rpeaks)})',
                  hovermode='x unified')
fig.show()

Частота дискретизации: 25.00 Гц
Длина записи: 1087.2 с (27180 отсчётов)

Подбор tkeo_factor (авто‑фильтр высоты):
  tkeo_factor=0.20  →  пиков:  502  ЧСС≈71 уд/мин
  tkeo_factor=0.25  →  пиков:  382  ЧСС≈52 уд/мин
  tkeo_factor=0.30  →  пиков:  320  ЧСС≈16 уд/мин
  tkeo_factor=0.35  →  пиков:  268  ЧСС≈11 уд/мин
  tkeo_factor=0.40  →  пиков:  186  ЧСС≈10 уд/мин
  tkeo_factor=0.50  →  пиков:  173  ЧСС≈10 уд/мин
  tkeo_factor=0.60  →  пиков:  171  ЧСС≈10 уд/мин
  tkeo_factor=0.70  →  пиков:  171  ЧСС≈10 уд/мин
  tkeo_factor=0.80  →  пиков:  167  ЧСС≈10 уд/мин

Визуализация: tkeo_factor=0.8, найдено 166 R‑пиков


In [ ]:
# -*- coding: utf-8 -*-
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks, savgol_filter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ─── НАСТРОЙКИ ─────────────────────────────────────────────
DATA_FILE = 'data4ch_0_6.xls'      # <-- ваш файл
DISTANCE_M = 0.565
BEST_TKEO = 0.8
MIN_RR_SEC = 0.4

# ─── ЗАГРУЗКА ──────────────────────────────────────────────
df = pd.read_csv(DATA_FILE, header=None)
time = df.iloc[:, 0].values.astype(float)
ch_chest_raw = df.iloc[:, 1].values.astype(float)
ch_wrist_raw = df.iloc[:, 2].values.astype(float)
ecg_raw = df.iloc[:, 4].values.astype(float)

fs = 1.0 / np.median(np.diff(time))
print(f"Частота дискретизации: {fs:.2f} Гц")

# ═════════════════════════════════════════════════════════
# 1. ПРЕДОБРАБОТКА ЭКГ (ваш метод, без изменений)
# ═════════════════════════════════════════════════════════
def butter_bandpass(data, low, high, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, min(high/nyq, 0.99)], btype='band')
    return filtfilt(b, a, data)

def preprocess_ecg_safe(raw, fs):
    ecg = butter_bandpass(raw, 0.5, min(10.0, fs*0.45), fs)
    return (ecg - ecg.mean()) / (ecg.std() + 1e-8)

ecg_filt = preprocess_ecg_safe(ecg_raw, fs)

# Детектор R-пиков (ваш, копия)
def detect_rpeaks_robust(ecg, fs, min_rr_sec=0.5, tkeo_factor=0.3):
    pos = np.max(ecg) - np.median(ecg)
    neg = np.median(ecg) - np.min(ecg)
    inverted = neg > pos
    ecg_proc = -ecg if inverted else ecg

    tkeo = ecg_proc[1:-1]**2 - ecg_proc[:-2] * ecg_proc[2:]
    tkeo = np.insert(tkeo, 0, 0)
    win = max(3, int(0.05 * fs) | 1)
    tkeo_smooth = savgol_filter(tkeo, window_length=win, polyorder=2)
    thresh_tkeo = tkeo_factor * np.percentile(tkeo_smooth, 98)
    min_dist = int(min_rr_sec * fs)
    candidates, _ = find_peaks(tkeo_smooth, height=thresh_tkeo, distance=min_dist)

    half_win = max(1, int(0.03 * fs))
    peak_indices = []
    peak_heights = []
    for p in candidates:
        lo = max(0, p - half_win)
        hi = min(len(ecg), p + half_win + 1)
        local_peak = lo + (np.argmin(ecg[lo:hi]) if inverted else np.argmax(ecg[lo:hi]))
        win_bl = int(1.0 * fs)
        bl_start = max(0, local_peak - win_bl)
        bl_end = min(len(ecg), local_peak + win_bl)
        baseline = np.median(ecg[bl_start:bl_end])
        height = np.abs(ecg[local_peak] - baseline)
        peak_indices.append(local_peak)
        peak_heights.append(height)

    peak_indices = np.array(peak_indices)
    peak_heights = np.array(peak_heights)
    if len(peak_heights) < 2:
        return peak_indices
    height_threshold = 0.5 * np.median(peak_heights)
    keep = peak_heights >= height_threshold
    return np.unique(peak_indices[keep])

rpeaks = detect_rpeaks_robust(ecg_filt, fs, min_rr_sec=MIN_RR_SEC, tkeo_factor=BEST_TKEO)
print(f"Найдено {len(rpeaks)} R‑пиков")

# ═════════════════════════════════════════════════════════
# 2. УПРОЩЁННАЯ ОБРАБОТКА ФПГ
# ═════════════════════════════════════════════════════════
def preprocess_ppg_simple(raw, fs, low=0.5, high=5.0):
    # Полоса 0.5–5 Гц
    sig = butter_bandpass(raw, low, min(high, fs*0.45), fs)
    # min-max нормализация
    sig = (sig - sig.min()) / (sig.max() - sig.min() + 1e-9)
    return sig

ppg_chest = preprocess_ppg_simple(ch_chest_raw, fs)
ppg_arm   = preprocess_ppg_simple(ch_wrist_raw, fs)

# ═════════════════════════════════════════════════════════
# 3. ДЕТЕКЦИЯ FEET ОТНОСИТЕЛЬНО R-ПИКОВ (надёжнее)
# ═════════════════════════════════════════════════════════
def detect_foot_from_rpeak(ppg, rpeaks, fs, search_window_ms=(50, 400)):
    """
    Для каждого R-пика ищет foot (основание) пульсовой волны
    в окне [rpeak + search_window_ms[0] мс : rpeak + search_window_ms[1] мс].
    Возвращает индексы feet.
    """
    start_dt = int(search_window_ms[0] * fs / 1000)
    end_dt   = int(search_window_ms[1] * fs / 1000)
    feet = []
    for r in rpeaks:
        lo = r + start_dt
        hi = r + end_dt
        if hi >= len(ppg):
            continue
        seg = ppg[lo:hi]
        # Ищем минимум (основание) перед подъёмом
        foot_local = np.argmin(seg)
        feet.append(lo + foot_local)
    return np.array(feet, dtype=int)

feet_chest = detect_foot_from_rpeak(ppg_chest, rpeaks, fs, search_window_ms=(50, 400))
feet_arm   = detect_foot_from_rpeak(ppg_arm,   rpeaks, fs, search_window_ms=(50, 400))

print(f"Feet грудь: {len(feet_chest)}, Feet рука: {len(feet_arm)}")

# ═════════════════════════════════════════════════════════
# 4. РАСЧЁТ PTT (разница между feet руки и груди)
# ═════════════════════════════════════════════════════════
def compute_ptt_from_feet(feet_chest, feet_arm, fs, ptt_min_ms=20, ptt_max_ms=200):
    ptt_vals = []
    valid_chest_idx = []
    valid_arm_idx = []
    for fc in feet_chest:
        # Ищем ближайший foot на руке в пределах [fc + ptt_min, fc + ptt_max]
        candidates = feet_arm[(feet_arm >= fc + int(ptt_min_ms * fs / 1000)) &
                              (feet_arm <= fc + int(ptt_max_ms * fs / 1000))]
        if len(candidates) == 0:
            continue
        nearest = candidates[0]
        ptt = (nearest - fc) / fs
        ptt_vals.append(ptt)
        valid_chest_idx.append(fc)
        valid_arm_idx.append(nearest)
    ptt_vals = np.array(ptt_vals)
    valid_chest_idx = np.array(valid_chest_idx, dtype=int)
    valid_arm_idx = np.array(valid_arm_idx, dtype=int)
    if len(ptt_vals) == 0:
        return [], [], []
    # MAD-фильтр
    median_ptt = np.median(ptt_vals)
    mad = np.median(np.abs(ptt_vals - median_ptt))
    if mad == 0:
        valid = np.ones(len(ptt_vals), dtype=bool)
    else:
        z = np.abs(ptt_vals - median_ptt) / (mad * 1.4826)
        valid = z < 3.5
    return ptt_vals[valid], valid_chest_idx[valid], valid_arm_idx[valid]

ptt_valid, chest_feet_used, arm_feet_used = compute_ptt_from_feet(
    feet_chest, feet_arm, fs, ptt_min_ms=20, ptt_max_ms=200)

if len(ptt_valid) > 0:
    print(f"Валидных PTT: {len(ptt_valid)}")
    print(f"PTT (медиана): {np.median(ptt_valid)*1000:.1f} мс")
    print(f"PWV (медиана): {np.median(DISTANCE_M/ptt_valid):.2f} м/с")
else:
    print("Нет валидных PTT")

# ═════════════════════════════════════════════════════════
# ИНТЕРАКТИВНАЯ ВИЗУАЛИЗАЦИЯ С RANGESLIDER
# ═════════════════════════════════════════════════════════
fig = make_subplots(rows=3, cols=1,
                    shared_xaxes=True,
                    subplot_titles=('ЭКГ + R‑пики',
                                    'ФПГ грудь (фильтр 0.5-5 Гц)',
                                    'ФПГ рука (фильтр 0.5-5 Гц)'),
                    vertical_spacing=0.06)

# ЭКГ
fig.add_trace(go.Scattergl(x=time, y=ecg_filt, name='ЭКГ фильтр.',
                           line=dict(color='blue', width=0.8)), row=1, col=1)
fig.add_trace(go.Scattergl(x=time[rpeaks], y=ecg_filt[rpeaks],
                           mode='markers', name='R‑пики',
                           marker=dict(color='red', size=6, symbol='x')),
              row=1, col=1)

# ФПГ грудь
fig.add_trace(go.Scattergl(x=time, y=ppg_chest, name='ФПГ грудь',
                           line=dict(color='green', width=0.8)), row=2, col=1)
fig.add_trace(go.Scattergl(x=time[feet_chest], y=ppg_chest[feet_chest],
                           mode='markers', name='Feet грудь',
                           marker=dict(color='darkgreen', size=8, symbol='circle-open')),
              row=2, col=1)

# ФПГ рука
fig.add_trace(go.Scattergl(x=time, y=ppg_arm, name='ФПГ рука',
                           line=dict(color='purple', width=0.8)), row=3, col=1)
fig.add_trace(go.Scattergl(x=time[feet_arm], y=ppg_arm[feet_arm],
                           mode='markers', name='Feet рука',
                           marker=dict(color='darkviolet', size=8, symbol='circle-open')),
              row=3, col=1)

# Добавим линии для нескольких первых валидных PTT (чтобы проверить)
if len(ptt_valid) > 0:
    for i in range(min(5, len(ptt_valid))):
        idx_chest = chest_feet_used[i]
        idx_arm = arm_feet_used[i]
        fig.add_trace(go.Scattergl(x=[time[idx_chest], time[idx_arm]],
                                   y=[ppg_chest[idx_chest], ppg_arm[idx_arm]],
                                   mode='lines+markers',
                                   line=dict(color='black', dash='dot', width=1),
                                   marker=dict(size=5, color='black'),
                                   showlegend=(i==0),
                                   name=f'PTT={ptt_valid[i]*1000:.0f}мс'),
                      row=2, col=1)

# Включаем rangeslider на нижней оси
fig.update_xaxes(rangeslider=dict(visible=True), row=3, col=1)
fig.update_xaxes(title_text='Время, с', row=3, col=1)
fig.update_yaxes(title_text='Z‑score', row=1, col=1)
fig.update_yaxes(title_text='Норм.', row=2, col=1)
fig.update_yaxes(title_text='Норм.', row=3, col=1)

fig.update_layout(height=800, title='Диагностика R‑пиков и feet ФПГ (упрощённая обработка)',
                  hovermode='x unified')
fig.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# -*- coding: utf-8 -*-
"""
Пайплайн расчёта скорости распространения пульсовой волны (СРПВ)
с использованием детектора R-пиков (TKEO) и метода касательных
для поиска оснований (feet) пульсовой волны на сырых сигналах ФПГ.
Без разрушительной фильтрации.
"""

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks, savgol_filter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ─── НАСТРОЙКИ (подставьте свои) ───────────────────────
DATA_FILE = 'data4ch_0_6.xls'     # файл с данными
DISTANCE_M = 0.565                # расстояние грудь-запястье, м
BEST_TKEO = 0.8                   # порог TKEO для R-пиков
MIN_RR_SEC = 0.4                  # минимальный RR-интервал (сек)

# Параметры окна поиска foot после R-пика (мс)
FOOT_SEARCH_START_MS = 50
FOOT_SEARCH_END_MS   = 400

# Параметры для метода касательных
DERIV_WINDOW = 5        # окно Савицкого-Голея для производной (точки)
POLYORDER = 2           # порядок полинома

# Параметры PTT
PTT_MIN_MS = 20         # минимально возможный PTT (мс)
PTT_MAX_MS = 200        # максимально возможный PTT (мс)

# ─── ЗАГРУЗКА ДАННЫХ ─────────────────────────────────
print("Загрузка данных...")
df = pd.read_csv(DATA_FILE, header=None)
time = df.iloc[:, 0].values.astype(float)
ch_chest_raw = df.iloc[:, 1].values.astype(float)
ch_wrist_raw = df.iloc[:, 2].values.astype(float)
ecg_raw = df.iloc[:, 4].values.astype(float)

fs = 1.0 / np.median(np.diff(time))
print(f"Частота дискретизации: {fs:.2f} Гц")
print(f"Длина записи: {time[-1]-time[0]:.1f} с")

# ═════════════════════════════════════════════════════
# 1. ДЕТЕКЦИЯ R-ПИКОВ (ваш метод)
# ═════════════════════════════════════════════════════
def butter_bandpass(data, low, high, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, min(high/nyq, 0.99)], btype='band')
    return filtfilt(b, a, data)

def preprocess_ecg_safe(raw, fs):
    ecg = butter_bandpass(raw, 0.5, min(10.0, fs*0.45), fs)
    return (ecg - ecg.mean()) / (ecg.std() + 1e-8)

ecg_filt = preprocess_ecg_safe(ecg_raw, fs)

# Ваш детектор R-пиков (без изменений)
def detect_rpeaks_robust(ecg, fs, min_rr_sec=0.5, tkeo_factor=0.3):
    pos = np.max(ecg) - np.median(ecg)
    neg = np.median(ecg) - np.min(ecg)
    inverted = neg > pos
    ecg_proc = -ecg if inverted else ecg

    tkeo = ecg_proc[1:-1]**2 - ecg_proc[:-2] * ecg_proc[2:]
    tkeo = np.insert(tkeo, 0, 0)
    win = max(3, int(0.05 * fs) | 1)
    tkeo_smooth = savgol_filter(tkeo, window_length=win, polyorder=2)
    thresh_tkeo = tkeo_factor * np.percentile(tkeo_smooth, 98)
    min_dist = int(min_rr_sec * fs)
    candidates, _ = find_peaks(tkeo_smooth, height=thresh_tkeo, distance=min_dist)

    half_win = max(1, int(0.03 * fs))
    peak_indices = []
    peak_heights = []
    for p in candidates:
        lo = max(0, p - half_win)
        hi = min(len(ecg), p + half_win + 1)
        local_peak = lo + (np.argmin(ecg[lo:hi]) if inverted else np.argmax(ecg[lo:hi]))
        win_bl = int(1.0 * fs)
        bl_start = max(0, local_peak - win_bl)
        bl_end = min(len(ecg), local_peak + win_bl)
        baseline = np.median(ecg[bl_start:bl_end])
        height = np.abs(ecg[local_peak] - baseline)
        peak_indices.append(local_peak)
        peak_heights.append(height)

    peak_indices = np.array(peak_indices)
    peak_heights = np.array(peak_heights)
    if len(peak_heights) < 2:
        return peak_indices
    height_threshold = 0.5 * np.median(peak_heights)
    keep = peak_heights >= height_threshold
    return np.unique(peak_indices[keep])

print("Детекция R-пиков...")
rpeaks = detect_rpeaks_robust(ecg_filt, fs, min_rr_sec=MIN_RR_SEC, tkeo_factor=BEST_TKEO)
print(f"Найдено R-пиков: {len(rpeaks)}")

# ═════════════════════════════════════════════════════
# 2. ДЕТЕКЦИЯ ОСНОВАНИЙ (FOOT) ПУЛЬСОВОЙ ВОЛНЫ
#    МЕТОДОМ КАСАТЕЛЬНЫХ НА СЫРОМ СИГНАЛЕ
# ═════════════════════════════════════════════════════
def detect_foot_intersecting_tangents(signal_raw, rpeaks, fs,
                                      search_start_ms=50, search_end_ms=400,
                                      deriv_window=5, polyorder=2):
    """
    Для каждого R-пика ищет foot (основание пульсовой волны) в окне
    [R + start_ms, R + end_ms] методом пересекающихся касательных
    на СЫРОМ сигнале.
    """
    feet = []
    start_dt = int(search_start_ms * fs / 1000)
    end_dt   = int(search_end_ms * fs / 1000)

    for r in rpeaks:
        lo = r + start_dt
        hi = r + end_dt
        if hi >= len(signal_raw):
            continue
        segment = signal_raw[lo:hi]                    # сырой фрагмент
        if len(segment) < deriv_window:
            continue

        # 1. Минимум как базовая линия
        min_idx = np.argmin(segment)
        min_val = segment[min_idx]

        # 2. Сглаженная производная (устойчивость к шуму)
        deriv = savgol_filter(segment, deriv_window, polyorder, deriv=1)

        # 3. Максимум производной (точка максимального подъёма)
        max_deriv_idx = np.argmax(deriv)
        if max_deriv_idx >= len(segment) or max_deriv_idx < 0:
            continue
        max_deriv_val = segment[max_deriv_idx]
        slope = deriv[max_deriv_idx]

        # 4. Пересечение касательной с уровнем минимума
        #    y = slope * (x - max_deriv_idx) + max_deriv_val
        #    min_val = slope * (x_int - max_deriv_idx) + max_deriv_val
        if abs(slope) < 1e-9:
            continue
        x_int = max_deriv_idx + (min_val - max_deriv_val) / slope

        # 5. Пересечение должно быть внутри сегмента и до пика
        if 0 <= x_int < len(segment):
            foot_idx = lo + int(round(x_int))
            feet.append(foot_idx)

    return np.array(feet, dtype=int)

print("Поиск оснований (feet) методом касательных...")
feet_chest = detect_foot_intersecting_tangents(
    ch_chest_raw, rpeaks, fs,
    search_start_ms=FOOT_SEARCH_START_MS,
    search_end_ms=FOOT_SEARCH_END_MS,
    deriv_window=DERIV_WINDOW, polyorder=POLYORDER
)
feet_arm = detect_foot_intersecting_tangents(
    ch_wrist_raw, rpeaks, fs,
    search_start_ms=FOOT_SEARCH_START_MS,
    search_end_ms=FOOT_SEARCH_END_MS,
    deriv_window=DERIV_WINDOW, polyorder=POLYORDER
)
print(f"Feet грудь: {len(feet_chest)}, Feet рука: {len(feet_arm)}")

# ═════════════════════════════════════════════════════
# 3. РАСЧЁТ PTT И PWV
# ═════════════════════════════════════════════════════
def compute_ptt_and_pwv(feet_chest, feet_arm, fs, distance_m,
                        ptt_min_ms=20, ptt_max_ms=200):
    """Сопоставляет foot груди и руки, отбрасывая нефизиологичные пары."""
    ptt_list = []
    chest_used = []
    arm_used = []
    for fc in feet_chest:
        # Ищем ближайший foot на руке в допустимом окне
        candidates = feet_arm[(feet_arm >= fc + int(ptt_min_ms * fs / 1000)) &
                              (feet_arm <= fc + int(ptt_max_ms * fs / 1000))]
        if len(candidates) == 0:
            continue
        nearest = candidates[0]
        ptt = (nearest - fc) / fs  # сек
        ptt_list.append(ptt)
        chest_used.append(fc)
        arm_used.append(nearest)

    ptt_arr = np.array(ptt_list)
    chest_arr = np.array(chest_used, dtype=int)
    arm_arr = np.array(arm_used, dtype=int)
    if len(ptt_arr) == 0:
        return [], [], [], []

    # MAD-фильтр для удаления выбросов PTT
    median_ptt = np.median(ptt_arr)
    mad = np.median(np.abs(ptt_arr - median_ptt))
    if mad == 0:
        valid = np.ones(len(ptt_arr), dtype=bool)
    else:
        z = np.abs(ptt_arr - median_ptt) / (mad * 1.4826)
        valid = z < 3.5

    ptt_valid = ptt_arr[valid]
    chest_valid = chest_arr[valid]
    arm_valid = arm_arr[valid]
    pwv_valid = distance_m / ptt_valid

    return ptt_valid, chest_valid, arm_valid, pwv_valid

ptt_valid, chest_feet_used, arm_feet_used, pwv_valid = compute_ptt_and_pwv(
    feet_chest, feet_arm, fs, DISTANCE_M,
    ptt_min_ms=PTT_MIN_MS, ptt_max_ms=PTT_MAX_MS
)

if len(ptt_valid) > 0:
    print(f"\nРезультаты PTT/PWV:")
    print(f"  Валидных пар: {len(ptt_valid)}")
    print(f"  PTT (медиана): {np.median(ptt_valid)*1000:.1f} мс")
    print(f"  PWV (медиана): {np.median(pwv_valid):.2f} м/с")
    print(f"  PTT диапазон : {ptt_valid.min()*1000:.0f}–{ptt_valid.max()*1000:.0f} мс")
else:
    print("Не найдено ни одной валидной пары PTT. Проверьте параметры окон.")

# ═════════════════════════════════════════════════════
# 4. ИНТЕРАКТИВНАЯ ВИЗУАЛИЗАЦИЯ (с rangeslider)
# ═════════════════════════════════════════════════════
print("\nПостроение интерактивных графиков...")
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=('ЭКГ + R‑пики',
                    'Сигнал груди (сырой) + foot',
                    'Сигнал руки (сырой) + foot'),
    vertical_spacing=0.07
)

# ЭКГ
fig.add_trace(go.Scattergl(
    x=time, y=ecg_filt, name='ЭКГ (фильтр.)',
    line=dict(color='blue', width=0.7)), row=1, col=1)
fig.add_trace(go.Scattergl(
    x=time[rpeaks], y=ecg_filt[rpeaks],
    mode='markers', name='R‑пики',
    marker=dict(color='red', size=6, symbol='x')), row=1, col=1)

# Грудь (сырой)
fig.add_trace(go.Scattergl(
    x=time, y=ch_chest_raw, name='Грудь (сырой)',
    line=dict(color='green', width=0.7)), row=2, col=1)
fig.add_trace(go.Scattergl(
    x=time[feet_chest], y=ch_chest_raw[feet_chest],
    mode='markers', name='Foot грудь',
    marker=dict(color='darkgreen', size=8, symbol='circle-open')), row=2, col=1)

# Рука (сырой)
fig.add_trace(go.Scattergl(
    x=time, y=ch_wrist_raw, name='Рука (сырой)',
    line=dict(color='purple', width=0.7)), row=3, col=1)
fig.add_trace(go.Scattergl(
    x=time[feet_arm], y=ch_wrist_raw[feet_arm],
    mode='markers', name='Foot рука',
    marker=dict(color='darkviolet', size=8, symbol='circle-open')), row=3, col=1)

# Показываем первые несколько соединительных линий PTT
if len(ptt_valid) > 0:
    for i in range(min(5, len(ptt_valid))):
        fc = chest_feet_used[i]
        fa = arm_feet_used[i]
        fig.add_trace(go.Scattergl(
            x=[time[fc], time[fa]],
            y=[ch_chest_raw[fc], ch_wrist_raw[fa]],
            mode='lines+markers',
            line=dict(color='black', dash='dot', width=1),
            marker=dict(size=5, color='black'),
            showlegend=(i == 0),
            name=f'PTT={ptt_valid[i]*1000:.0f} мс'
        ), row=2, col=1)

fig.update_xaxes(rangeslider=dict(visible=True), row=3, col=1)
fig.update_xaxes(title_text='Время, с', row=3, col=1)
fig.update_yaxes(title_text='Z‑score', row=1, col=1)
fig.update_yaxes(title_text='Отсчёты АЦП', row=2, col=1)
fig.update_yaxes(title_text='Отсчёты АЦП', row=3, col=1)

fig.update_layout(
    height=800,
    title='Детекция оснований (foot) пульсовой волны методом касательных',
    hovermode='x unified'
)
fig.show()

# ═════════════════════════════════════════════════════
# 5. ФОРМИРОВАНИЕ ML-ДАТАСЕТА (окна вокруг foot груди)
# ═════════════════════════════════════════════════════
if len(ptt_valid) > 10:  # минимальное количество для обучения
    print("\nФормирование датасета для ML...")

    # Окно захватывает 0.2 с до foot и 0.6 с после
    win_before = int(0.2 * fs)
    win_after  = int(0.6 * fs)
    win_len = win_before + win_after

    X, y = [], []
    for fc, pwv in zip(chest_feet_used, pwv_valid):
        if fc - win_before < 0 or fc + win_after >= len(ch_chest_raw):
            continue
        # Берём сегменты сырых сигналов
        seg_chest = ch_chest_raw[fc - win_before : fc + win_after]
        seg_arm   = ch_wrist_raw[fc - win_before : fc + win_after]
        # Нормализация min-max (сохраняет форму)
        seg_chest = (seg_chest - seg_chest.min()) / (seg_chest.max() - seg_chest.min() + 1e-9)
        seg_arm   = (seg_arm - seg_arm.min()) / (seg_arm.max() - seg_arm.min() + 1e-9)
        X.append(np.column_stack([seg_chest, seg_arm]))
        y.append(pwv)

    X = np.array(X)   # (N, win_len, 2)
    y = np.array(y)
    print(f"Обучающих примеров: {len(X)}, форма окна: {X.shape[1:]}")

    # ═════════════════════════════════════════════════
    # 6. ОБУЧЕНИЕ 1D CNN ДЛЯ ПРЕДСКАЗАНИЯ PWV
    # ═════════════════════════════════════════════════
    from sklearn.model_selection import train_test_split
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = Sequential([
        Conv1D(32, kernel_size=5, activation='relu', input_shape=(win_len, 2)),
        MaxPooling1D(2),
        Conv1D(64, kernel_size=3, activation='relu'),
        MaxPooling1D(2),
        Flatten(),
        Dropout(0.5),
        Dense(32, activation='relu'),
        Dense(1)  # одно значение PWV
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    print("\nОбучение CNN...")
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test),
                        epochs=80, batch_size=8, verbose=1)

    model.save('pwv_cnn_model_raw.h5')
    print("Модель сохранена: pwv_cnn_model_raw.h5")

    # Оценка
    pred = model.predict(X_test).flatten()
    mae = np.mean(np.abs(pred - y_test))
    print(f"\nMAE на тестовой выборке: {mae:.2f} м/с")
    print(f"Медианная PWV (тест): {np.median(y_test):.2f} м/с")
else:
    print("Недостаточно валидных PTT для обучения (нужно >10).")

Загрузка данных...
Частота дискретизации: 25.00 Гц
Длина записи: 1087.2 с
Детекция R-пиков...
Найдено R-пиков: 168
Поиск оснований (feet) методом касательных...
Feet грудь: 86, Feet рука: 108

Результаты PTT/PWV:
  Валидных пар: 27
  PTT (медиана): 0.0 мс
  PWV (медиана): inf м/с
  PTT диапазон : 0–200 мс

Построение интерактивных графиков...


/tmp/ipykernel_20649/929960823.py:208: RuntimeWarning:

divide by zero encountered in divide



/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.




Формирование датасета для ML...
Обучающих примеров: 27, форма окна: (20, 2)

Обучение CNN...
Epoch 1/80
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 218ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 2/80
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 3/80
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 4/80
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 5/80
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 6/80
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 7/80
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 8/80
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 9/80
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - loss: nan - mae: nan - val_los

Модель сохранена: pwv_cnn_model_raw.h5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step

MAE на тестовой выборке: nan м/с
Медианная PWV (тест): 14.13 м/с


In [ ]:
pip install pyPPG

In [ ]:
# -*- coding: utf-8 -*-
"""
Полный пайплайн: R-пики → foot (pyPPG) → PTT → PWV → CNN.
Без фильтрации сигналов датчиков Холла.
"""

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks, savgol_filter
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pyPPG  # <-- основной инструмент для анализа PPG-подобных сигналов

# ═══════════════════════════════════════════════════════
# НАСТРОЙКИ (подставьте свои значения)
# ═══════════════════════════════════════════════════════
DATA_FILE = 'data4ch_0_6.xls'        # ваш CSV-файл
DISTANCE_M = 0.565                   # расстояние грудь–запястье, м
BEST_TKEO = 0.8                      # ваш оптимальный порог TKEO
MIN_RR_SEC = 0.4                     # минимальный RR-интервал, с

# Окно поиска foot после R-пика (мс)
FOOT_START_MS = 50
FOOT_END_MS   = 400

# Допустимый диапазон PTT (мс)
PTT_MIN_MS = 20
PTT_MAX_MS = 200

# ═══════════════════════════════════════════════════════
# 1. ЗАГРУЗКА ДАННЫХ
# ═══════════════════════════════════════════════════════
print("Загрузка данных...")
df = pd.read_csv(DATA_FILE, header=None)
time = df.iloc[:, 0].values.astype(float)
ch_chest_raw = df.iloc[:, 1].values.astype(float)
ch_wrist_raw = df.iloc[:, 2].values.astype(float)
ecg_raw = df.iloc[:, 4].values.astype(float)

fs = 1.0 / np.median(np.diff(time))
print(f"Частота дискретизации: {fs:.2f} Гц")
print(f"Длительность: {time[-1]-time[0]:.1f} с")

# ═══════════════════════════════════════════════════════
# 2. ДЕТЕКЦИЯ R-ПИКОВ (ваш метод, без изменений)
# ═══════════════════════════════════════════════════════
def butter_bandpass(data, low, high, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, min(high/nyq, 0.99)], btype='band')
    return filtfilt(b, a, data)

def preprocess_ecg_safe(raw, fs):
    ecg = butter_bandpass(raw, 0.5, min(10.0, fs*0.45), fs)
    return (ecg - ecg.mean()) / (ecg.std() + 1e-8)

ecg_filt = preprocess_ecg_safe(ecg_raw, fs)

def detect_rpeaks_robust(ecg, fs, min_rr_sec=0.5, tkeo_factor=0.3):
    pos = np.max(ecg) - np.median(ecg)
    neg = np.median(ecg) - np.min(ecg)
    inverted = neg > pos
    ecg_proc = -ecg if inverted else ecg

    tkeo = ecg_proc[1:-1]**2 - ecg_proc[:-2] * ecg_proc[2:]
    tkeo = np.insert(tkeo, 0, 0)
    win = max(3, int(0.05 * fs) | 1)
    tkeo_smooth = savgol_filter(tkeo, window_length=win, polyorder=2)
    thresh_tkeo = tkeo_factor * np.percentile(tkeo_smooth, 98)
    min_dist = int(min_rr_sec * fs)
    candidates, _ = find_peaks(tkeo_smooth, height=thresh_tkeo, distance=min_dist)

    half_win = max(1, int(0.03 * fs))
    peak_indices = []
    peak_heights = []
    for p in candidates:
        lo = max(0, p - half_win)
        hi = min(len(ecg), p + half_win + 1)
        local_peak = lo + (np.argmin(ecg[lo:hi]) if inverted else np.argmax(ecg[lo:hi]))
        win_bl = int(1.0 * fs)
        bl_start = max(0, local_peak - win_bl)
        bl_end = min(len(ecg), local_peak + win_bl)
        baseline = np.median(ecg[bl_start:bl_end])
        height = np.abs(ecg[local_peak] - baseline)
        peak_indices.append(local_peak)
        peak_heights.append(height)

    peak_indices = np.array(peak_indices)
    peak_heights = np.array(peak_heights)
    if len(peak_heights) < 2:
        return peak_indices
    height_threshold = 0.5 * np.median(peak_heights)
    keep = peak_heights >= height_threshold
    return np.unique(peak_indices[keep])

print("Поиск R-пиков...")
rpeaks = detect_rpeaks_robust(ecg_filt, fs, min_rr_sec=MIN_RR_SEC, tkeo_factor=BEST_TKEO)
print(f"Найдено R-пиков: {len(rpeaks)}")



from scipy.interpolate import interp1d

def remove_zero_spikes(sig, fs):
    """
    Заменяет точки, равные 0, линейной интерполяцией по соседним.
    Если несколько нулей подряд, интерполирует весь участок.
    """
    sig = sig.astype(float).copy()
    bad = (sig == 0)
    if not np.any(bad):
        return sig
    # Заменяем нули на NaN, затем интерполируем
    sig[bad] = np.nan
    x = np.arange(len(sig))
    valid = ~np.isnan(sig)
    # Если первый или последний элемент NaN, заполняем ближайшим значением
    if not valid[0]:
        first_valid = np.argmax(valid)
        sig[:first_valid] = sig[first_valid]
    if not valid[-1]:
        last_valid = len(sig) - 1 - np.argmax(valid[::-1])
        sig[last_valid:] = sig[last_valid]
    f = interp1d(x[valid], sig[valid], kind='linear', fill_value='extrapolate')
    sig_clean = f(x)
    return sig_clean

def lowpass_filter(data, fs, cutoff=10.0, order=4):
    """Фильтр низких частот (сохраняет частоты ниже cutoff)"""
    nyq = 0.5 * fs
    cutoff = min(cutoff, nyq * 0.99)
    b, a = butter(order, cutoff / nyq, btype='low')
    return filtfilt(b, a, data)

def preprocess_ppg_light(raw, fs):
    """Лёгкая очистка: удаление нулей, lowpass 10 Гц, min‑max нормализация."""
    sig = remove_zero_spikes(raw, fs)
    sig = lowpass_filter(sig, fs, cutoff=10.0)   # убираем частоты выше 10 Гц
    # Нормализация не обязательна, но для единообразия:
    sig = (sig - sig.min()) / (sig.max() - sig.min() + 1e-9)
    return sig



# ═══════════════════════════════════════════════════════
# 3. ДЕТЕКЦИЯ FOOT ПУЛЬСОВОЙ ВОЛНЫ С ПОМОЩЬЮ pyPPG
# ═══════════════════════════════════════════════════════
def detect_foot_pyPPG(signal_raw, rpeaks, fs,
                      search_start_ms=50, search_end_ms=400):
    start_dt = int(search_start_ms * fs / 1000)
    end_dt   = int(search_end_ms * fs / 1000)
    feet = []
    for r in rpeaks:
        lo = r + start_dt
        hi = r + end_dt
        if hi >= len(signal_raw):
            continue
        segment = signal_raw[lo:hi]
        if len(segment) < 3:   # слишком короткий сегмент
            continue

        # --- Пробуем pyPPG с корректным синтаксисом ---
        try:
            # Попытка 1: позиционный аргумент (fs без ключа)
            s = pyPPG.PPG(segment, fs)
        except TypeError:
            try:
                # Попытка 2: объект без fs, затем установка атрибута
                s = pyPPG.PPG(segment)
                s.fs = fs
            except Exception as e:
                print(f"pyPPG не удалось создать объект (индекс {r}): {e}")
                continue
        except Exception as e:
            print(f"Ошибка pyPPG (индекс {r}): {e}")
            continue

        # --- Анализ ---
        try:
            s.get_analysis()
        except Exception as e:
            print(f"Ошибка анализа pyPPG (индекс {r}): {e}")
            continue

        if hasattr(s, 'onsets') and len(s.onsets) > 0:
            onset_idx = s.onsets[0]
            if 0 <= onset_idx < len(segment):
                feet.append(lo + onset_idx)

    return np.array(feet, dtype=int)

print("Предобработка сигналов (удаление нулей + lowpass 10 Гц)...")
ppg_chest_clean = preprocess_ppg_light(ch_chest_raw, fs)
ppg_arm_clean   = preprocess_ppg_light(ch_wrist_raw, fs)

# Детекция foot на очищенных сигналах
feet_chest = detect_foot_pyPPG(ppg_chest_clean, rpeaks, fs,
                               search_start_ms=50, search_end_ms=400)
feet_arm   = detect_foot_pyPPG(ppg_arm_clean, rpeaks, fs,
                               search_start_ms=50, search_end_ms=400)

# ═══════════════════════════════════════════════════════
# 4. РАСЧЁТ PTT И PWV (foot-to-foot)
# ═══════════════════════════════════════════════════════
def compute_ptt_pwv(feet_chest, feet_arm, fs, distance_m,
                    ptt_min_ms=20, ptt_max_ms=200):
    ptt_list, chest_used, arm_used = [], [], []
    for fc in feet_chest:
        # Ищем ближайший foot руки в заданном временном окне
        candidates = feet_arm[(feet_arm >= fc + int(ptt_min_ms * fs / 1000)) &
                              (feet_arm <= fc + int(ptt_max_ms * fs / 1000))]
        if len(candidates) == 0:
            continue
        nearest = candidates[0]
        ptt = (nearest - fc) / fs
        ptt_list.append(ptt)
        chest_used.append(fc)
        arm_used.append(nearest)

    ptt_arr = np.array(ptt_list)
    chest_arr = np.array(chest_used, dtype=int)
    arm_arr = np.array(arm_used, dtype=int)

    if len(ptt_arr) == 0:
        return [], [], [], []

    # MAD-фильтр выбросов
    med_ptt = np.median(ptt_arr)
    mad = np.median(np.abs(ptt_arr - med_ptt))
    if mad == 0:
        valid = np.ones(len(ptt_arr), dtype=bool)
    else:
        z = np.abs(ptt_arr - med_ptt) / (mad * 1.4826)
        valid = z < 3.5

    ptt_valid = ptt_arr[valid]
    chest_valid = chest_arr[valid]
    arm_valid = arm_arr[valid]
    pwv_valid = distance_m / ptt_valid
    return ptt_valid, chest_valid, arm_valid, pwv_valid

ptt_val, chest_feet, arm_feet, pwv_val = compute_ptt_pwv(
    feet_chest, feet_arm, fs, DISTANCE_M,
    ptt_min_ms=PTT_MIN_MS, ptt_max_ms=PTT_MAX_MS)

if len(ptt_val) > 0:
    print(f"\nРезультаты:")
    print(f"  Валидных пар: {len(ptt_val)}")
    print(f"  PTT (медиана): {np.median(ptt_val)*1000:.1f} мс")
    print(f"  PWV (медиана): {np.median(pwv_val):.2f} м/с")
else:
    print("Валидных PTT не найдено. Проверьте окна детекции.")

# ═══════════════════════════════════════════════════════
# 5. ИНТЕРАКТИВНАЯ ВИЗУАЛИЗАЦИЯ (Plotly)
# ═══════════════════════════════════════════════════════
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    subplot_titles=('ЭКГ + R‑пики',
                    'Грудь (сырой) + foot (pyPPG)',
                    'Рука (сырой) + foot (pyPPG)'),
    vertical_spacing=0.07)

# ЭКГ
fig.add_trace(go.Scattergl(x=time, y=ecg_filt,
                           name='ЭКГ', line=dict(color='blue', width=0.7)),
              row=1, col=1)
fig.add_trace(go.Scattergl(x=time[rpeaks], y=ecg_filt[rpeaks],
                           mode='markers', name='R‑пики',
                           marker=dict(color='red', size=6, symbol='x')),
              row=1, col=1)

# Грудь
fig.add_trace(go.Scattergl(x=time, y=ch_chest_raw,
                           name='Грудь', line=dict(color='green', width=0.7)),
              row=2, col=1)
fig.add_trace(go.Scattergl(x=time[feet_chest], y=ch_chest_raw[feet_chest],
                           mode='markers', name='Foot грудь',
                           marker=dict(color='darkgreen', size=8, symbol='circle-open')),
              row=2, col=1)

# Рука
fig.add_trace(go.Scattergl(x=time, y=ch_wrist_raw,
                           name='Рука', line=dict(color='purple', width=0.7)),
              row=3, col=1)
fig.add_trace(go.Scattergl(x=time[feet_arm], y=ch_wrist_raw[feet_arm],
                           mode='markers', name='Foot рука',
                           marker=dict(color='darkviolet', size=8, symbol='circle-open')),
              row=3, col=1)

# Соединительные линии для первых 5 PTT
if len(ptt_val) > 0:
    for i in range(min(5, len(ptt_val))):
        fc = chest_feet[i]
        fa = arm_feet[i]
        fig.add_trace(go.Scattergl(
            x=[time[fc], time[fa]],
            y=[ch_chest_raw[fc], ch_wrist_raw[fa]],
            mode='lines+markers',
            line=dict(color='black', dash='dot', width=1),
            marker=dict(size=5, color='black'),
            showlegend=(i == 0),
            name=f'PTT={ptt_val[i]*1000:.0f} мс'
        ), row=2, col=1)

fig.update_xaxes(rangeslider=dict(visible=True), row=3, col=1)
fig.update_xaxes(title_text='Время, с', row=3, col=1)
fig.update_yaxes(title_text='Z‑score', row=1, col=1)
fig.update_yaxes(title_text='Отсчёты', row=2, col=1)
fig.update_yaxes(title_text='Отсчёты', row=3, col=1)
fig.update_layout(height=800, title='Детекция foot пульсовой волны (pyPPG)',
                  hovermode='x unified')
fig.show()

# ═══════════════════════════════════════════════════════
# 6. ОБУЧЕНИЕ CNN ДЛЯ ПРЕДСКАЗАНИЯ PWV (опционально)
# ═══════════════════════════════════════════════════════
if len(ptt_val) > 10:
    print("\nФормирование датасета для ML...")
    win_before = int(0.2 * fs)
    win_after  = int(0.6 * fs)
    win_len = win_before + win_after

    X, y = [], []
    for fc, pwv in zip(chest_feet, pwv_val):
        if fc - win_before < 0 or fc + win_after >= len(ch_chest_raw):
            continue
        seg_c = ch_chest_raw[fc - win_before : fc + win_after]
        seg_a = ch_wrist_raw[fc - win_before : fc + win_after]
        seg_c = (seg_c - seg_c.min()) / (seg_c.max() - seg_c.min() + 1e-9)
        seg_a = (seg_a - seg_a.min()) / (seg_a.max() - seg_a.min() + 1e-9)
        X.append(np.column_stack([seg_c, seg_a]))
        y.append(pwv)

    X = np.array(X, dtype='float32')
    y = np.array(y, dtype='float32')
    print(f"Обучающих примеров: {len(X)}")

    from sklearn.model_selection import train_test_split
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

    model = Sequential([
        Conv1D(32, kernel_size=5, activation='relu', input_shape=(win_len, 2)),
        MaxPooling1D(2),
        Conv1D(64, kernel_size=3, activation='relu'),
        MaxPooling1D(2),
        Flatten(),
        Dropout(0.5),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    print("Обучение модели...")
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test),
                        epochs=80, batch_size=8, verbose=1)
    model.save('pwv_cnn_pyPPG.h5')
    print("Модель сохранена.")

    pred = model.predict(X_test).flatten()
    mae = np.mean(np.abs(pred - y_test))
    print(f"MAE на тесте: {mae:.2f} м/с")
else:
    print("Недостаточно данных для обучения модели (нужно >10 циклов).")

Загрузка данных...
Частота дискретизации: 25.00 Гц
Длительность: 1087.2 с
Поиск R-пиков...
Найдено R-пиков: 168
Предобработка сигналов (удаление нулей + lowpass 10 Гц)...
pyPPG не удалось создать объект (индекс 138): 'numpy.ndarray' object has no attribute 'fs'
pyPPG не удалось создать объект (индекс 304): 'numpy.ndarray' object has no attribute 'fs'
pyPPG не удалось создать объект (индекс 481): 'numpy.ndarray' object has no attribute 'fs'
pyPPG не удалось создать объект (индекс 665): 'numpy.ndarray' object has no attribute 'fs'
pyPPG не удалось создать объект (индекс 854): 'numpy.ndarray' object has no attribute 'fs'
pyPPG не удалось создать объект (индекс 1041): 'numpy.ndarray' object has no attribute 'fs'
pyPPG не удалось создать объект (индекс 1068): 'numpy.ndarray' object has no attribute 'fs'
pyPPG не удалось создать объект (индекс 1215): 'numpy.ndarray' object has no attribute 'fs'
pyPPG не удалось создать объект (индекс 1383): 'numpy.ndarray' object has no attribute 'fs'
pyPPG 

Недостаточно данных для обучения модели (нужно >10 циклов).


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Загрузка ---
DATA_FILE = 'data4ch_0_6.xls'   # ваш файл
df = pd.read_csv(DATA_FILE, header=None)
time = df.iloc[:, 0].values.astype(float)
ch_chest_raw = df.iloc[:, 1].values.astype(float)
ch_wrist_raw = df.iloc[:, 2].values.astype(float)
ecg_raw = df.iloc[:, 4].values.astype(float)

fs = 1.0 / np.median(np.diff(time))
print(f"Частота дискретизации: {fs:.2f} Гц")

# --- Функция замены нулей на предыдущую точку ---
def remove_zeros_forward_fill(sig):
    """Заменяет нулевые значения на последнее ненулевое (вперёд)."""
    sig = sig.copy()
    for i in range(1, len(sig)):
        if sig[i] == 0:
            sig[i] = sig[i-1]
    # Если самый первый элемент 0, заполним ближайшим ненулевым
    if sig[0] == 0:
        first_nonzero = np.argmax(sig != 0)
        sig[0] = sig[first_nonzero] if first_nonzero < len(sig) else 0
    return sig

# Применяем к обоим каналам
ch_chest_clean = remove_zeros_forward_fill(ch_chest_raw)
ch_wrist_clean = remove_zeros_forward_fill(ch_wrist_raw)

# --- Интерактивная визуализация ---
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=('Канал груди (сырой vs очищенный)', 'Канал руки (сырой vs очищенный)'),
    vertical_spacing=0.1
)

# Грудь: исходный и очищенный
fig.add_trace(go.Scattergl(x=time, y=ch_chest_raw,
                           name='Грудь исходный', line=dict(color='lightgreen', width=0.6)),
              row=1, col=1)
fig.add_trace(go.Scattergl(x=time, y=ch_chest_clean,
                           name='Грудь очищенный', line=dict(color='darkgreen', width=1.2)),
              row=1, col=1)

# Рука: исходный и очищенный
fig.add_trace(go.Scattergl(x=time, y=ch_wrist_raw,
                           name='Рука исходный', line=dict(color='plum', width=0.6)),
              row=2, col=1)
fig.add_trace(go.Scattergl(x=time, y=ch_wrist_clean,
                           name='Рука очищенный', line=dict(color='purple', width=1.2)),
              row=2, col=1)

fig.update_xaxes(rangeslider=dict(visible=True), row=2, col=1)
fig.update_xaxes(title_text='Время, с', row=2, col=1)
fig.update_yaxes(title_text='Значение', row=1, col=1)
fig.update_yaxes(title_text='Значение', row=2, col=1)
fig.update_layout(height=600, title='Сравнение до и после замены нулевых выбросов',
                  hovermode='x unified')
fig.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
from scipy.signal import butter, filtfilt

def lowpass_filter(data, fs, cutoff=10.0, order=4):
    """Фильтр низких частот Баттерворта"""
    nyq = 0.5 * fs
    cutoff = min(cutoff, nyq * 0.99)   # гарантия < Найквиста
    b, a = butter(order, cutoff / nyq, btype='low')
    return filtfilt(b, a, data)

# Применяем к уже очищенным от нулей сигналам
ch_chest_lp = lowpass_filter(ch_chest_clean, fs, cutoff=10.0)
ch_wrist_lp = lowpass_filter(ch_wrist_clean, fs, cutoff=10.0)

# --- Интерактивная визуализация трёх этапов ---
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=('Канал груди', 'Канал руки'),
    vertical_spacing=0.1
)

# Грудь
fig.add_trace(go.Scattergl(x=time, y=ch_chest_raw,
                           name='Сырой', line=dict(color='lightgreen', width=0.5)),
              row=1, col=1)
fig.add_trace(go.Scattergl(x=time, y=ch_chest_clean,
                           name='Без нулей', line=dict(color='green', width=1.0)),
              row=1, col=1)
fig.add_trace(go.Scattergl(x=time, y=ch_chest_lp,
                           name='Lowpass 10 Гц', line=dict(color='darkgreen', width=1.5)),
              row=1, col=1)

# Рука
fig.add_trace(go.Scattergl(x=time, y=ch_wrist_raw,
                           name='Сырой', line=dict(color='plum', width=0.5)),
              row=2, col=1)
fig.add_trace(go.Scattergl(x=time, y=ch_wrist_clean,
                           name='Без нулей', line=dict(color='mediumorchid', width=1.0)),
              row=2, col=1)
fig.add_trace(go.Scattergl(x=time, y=ch_wrist_lp,
                           name='Lowpass 10 Гц', line=dict(color='purple', width=1.5)),
              row=2, col=1)

fig.update_xaxes(rangeslider=dict(visible=True), row=2, col=1)
fig.update_xaxes(title_text='Время, с', row=2, col=1)
fig.update_yaxes(title_text='Значение', row=1, col=1)
fig.update_yaxes(title_text='Значение', row=2, col=1)
fig.update_layout(height=600, title='Этапы очистки: сырой → без нулей → lowpass 10 Гц',
                  hovermode='x unified')
fig.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# -*- coding: utf-8 -*-
"""
Пайплайн: ЭКГ R-пики → foot (метод min‑before‑peak) → PTT → PWV → CNN.
Использует очищенные сигналы (замена нулей + lowpass 10 Гц).
"""

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks, savgol_filter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ═══════════════════════════════════════════════════════
# НАСТРОЙКИ
# ═══════════════════════════════════════════════════════
DATA_FILE = 'data4ch_0_5.xls'
DISTANCE_M = 0.565                 # расстояние грудь–запястье, м
BEST_TKEO = 0.4                    # ваш порог TKEO
MIN_RR_SEC = 0.4

FOOT_START_MS = 0                 # начало поиска foot после R-пика
FOOT_END_MS   = 400

PTT_MIN_MS = 20                    # допустимый диапазон PTT
PTT_MAX_MS = 200

# ═══════════════════════════════════════════════════════
# 1. ЗАГРУЗКА
# ═══════════════════════════════════════════════════════
print("Загрузка данных...")
df = pd.read_csv(DATA_FILE, header=None)
time = df.iloc[:, 0].values.astype(float)
time = time / 488 * 60
ch_chest_raw = df.iloc[:, 1].values.astype(float)
ch_wrist_raw = df.iloc[:, 2].values.astype(float)
ecg_raw = df.iloc[:, 4].values.astype(float)

fs = 1.0 / np.median(np.diff(time))
print(f"Частота дискретизации: {fs:.2f} Гц")

# ═══════════════════════════════════════════════════════
# 2. ДЕТЕКЦИЯ R-ПИКОВ (ваш метод)
# ═══════════════════════════════════════════════════════
def butter_bandpass(data, low, high, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, min(high/nyq, 0.99)], btype='band')
    return filtfilt(b, a, data)

def preprocess_ecg_safe(raw, fs):
    ecg = butter_bandpass(raw, 0.5, min(10.0, fs*0.45), fs)
    return (ecg - ecg.mean()) / (ecg.std() + 1e-8)

ecg_filt = preprocess_ecg_safe(ecg_raw, fs)

def detect_rpeaks_robust(ecg, fs, min_rr_sec=0.5, tkeo_factor=0.3):
    pos = np.max(ecg) - np.median(ecg)
    neg = np.median(ecg) - np.min(ecg)
    inverted = neg > pos
    ecg_proc = -ecg if inverted else ecg

    tkeo = ecg_proc[1:-1]**2 - ecg_proc[:-2] * ecg_proc[2:]
    tkeo = np.insert(tkeo, 0, 0)
    win = max(3, int(0.05 * fs) | 1)
    tkeo_smooth = savgol_filter(tkeo, window_length=win, polyorder=2)
    thresh_tkeo = tkeo_factor * np.percentile(tkeo_smooth, 98)
    min_dist = int(min_rr_sec * fs)
    candidates, _ = find_peaks(tkeo_smooth, height=thresh_tkeo, distance=min_dist)

    half_win = max(1, int(0.03 * fs))
    peak_indices = []
    peak_heights = []
    for p in candidates:
        lo = max(0, p - half_win)
        hi = min(len(ecg), p + half_win + 1)
        local_peak = lo + (np.argmin(ecg[lo:hi]) if inverted else np.argmax(ecg[lo:hi]))
        win_bl = int(1.0 * fs)
        bl_start = max(0, local_peak - win_bl)
        bl_end = min(len(ecg), local_peak + win_bl)
        baseline = np.median(ecg[bl_start:bl_end])
        height = np.abs(ecg[local_peak] - baseline)
        peak_indices.append(local_peak)
        peak_heights.append(height)

    peak_indices = np.array(peak_indices)
    peak_heights = np.array(peak_heights)
    if len(peak_heights) < 2:
        return peak_indices
    height_threshold = 0.5 * np.median(peak_heights)
    keep = peak_heights >= height_threshold
    return np.unique(peak_indices[keep])

print("Поиск R-пиков...")
rpeaks = detect_rpeaks_robust(ecg_filt, fs, min_rr_sec=MIN_RR_SEC, tkeo_factor=BEST_TKEO)
print(f"Найдено R-пиков: {len(rpeaks)}")

# ═══════════════════════════════════════════════════════
# 3. ОЧИСТКА СИГНАЛОВ ФПГ (замена нулей + lowpass)
# ═══════════════════════════════════════════════════════
def remove_zeros_forward_fill(sig):
    sig = sig.copy()
    for i in range(1, len(sig)):
        if sig[i] == 0:
            sig[i] = sig[i-1]
    if sig[0] == 0:
        first_nonzero = np.argmax(sig != 0)
        sig[0] = sig[first_nonzero] if first_nonzero < len(sig) else 0
    return sig

def lowpass_filter(data, fs, cutoff=10.0, order=4):
    nyq = 0.5 * fs
    cutoff = min(cutoff, nyq * 0.99)
    b, a = butter(order, cutoff / nyq, btype='low')
    return filtfilt(b, a, data)

print("Очистка сигналов...")
ch_chest_clean = remove_zeros_forward_fill(ch_chest_raw)
ch_wrist_clean = remove_zeros_forward_fill(ch_wrist_raw)
ch_chest_lp = lowpass_filter(ch_chest_clean, fs, cutoff=10.0)
ch_wrist_lp = lowpass_filter(ch_wrist_clean, fs, cutoff=10.0)

# ═══════════════════════════════════════════════════════
# 4. ДЕТЕКЦИЯ FOOT (min‑before‑peak)
# ═══════════════════════════════════════════════════════
def detect_foot_min_before_peak(signal_lp, rpeaks, fs,
                                search_start_ms=50, search_end_ms=400):
    start_dt = int(search_start_ms * fs / 1000)
    end_dt   = int(search_end_ms * fs / 1000)
    feet = []
    for r in rpeaks:
        lo = r + start_dt
        hi = r + end_dt
        if hi >= len(signal_lp):
            continue
        seg = signal_lp[lo:hi]
        if len(seg) < 3:
            continue
        # Систолический пик (максимум)
        peak_rel = np.argmax(seg)
        # Ищем минимум на участке от начала до пика
        search_end = max(1, int(peak_rel * 0.8))   # ограничим, чтобы не уйти в шум
        foot_rel = np.argmin(seg[:search_end])
        feet.append(lo + foot_rel)
    return np.array(feet, dtype=int)

print("Поиск foot...")
feet_chest = detect_foot_min_before_peak(ch_chest_lp, rpeaks, fs,
                                         FOOT_START_MS, FOOT_END_MS)
feet_arm   = detect_foot_min_before_peak(ch_wrist_lp, rpeaks, fs,
                                         FOOT_START_MS, FOOT_END_MS)
print(f"Feet грудь: {len(feet_chest)}, Feet рука: {len(feet_arm)}")

# ═══════════════════════════════════════════════════════
# 5. РАСЧЁТ PTT И PWV
# ═══════════════════════════════════════════════════════
def compute_ptt_pwv(feet_chest, feet_arm, fs, distance_m,
                    ptt_min_ms=20, ptt_max_ms=200):
    ptt_list, chest_idx, arm_idx = [], [], []
    for fc in feet_chest:
        candidates = feet_arm[(feet_arm >= fc + int(ptt_min_ms * fs / 1000)) &
                              (feet_arm <= fc + int(ptt_max_ms * fs / 1000))]
        if len(candidates) == 0:
            continue
        nearest = candidates[0]
        ptt = (nearest - fc) / fs
        ptt_list.append(ptt)
        chest_idx.append(fc)
        arm_idx.append(nearest)

    ptt_arr = np.array(ptt_list)
    chest_arr = np.array(chest_idx, dtype=int)
    arm_arr = np.array(arm_idx, dtype=int)

    if len(ptt_arr) == 0:
        return [], [], [], []

    # MAD-фильтр
    med = np.median(ptt_arr)
    mad = np.median(np.abs(ptt_arr - med))
    if mad == 0:
        valid = np.ones(len(ptt_arr), dtype=bool)
    else:
        z = np.abs(ptt_arr - med) / (mad * 1.4826)
        valid = z < 3.5

    ptt_valid = ptt_arr[valid]
    chest_valid = chest_arr[valid]
    arm_valid = arm_arr[valid]
    pwv_valid = distance_m / ptt_valid
    return ptt_valid, chest_valid, arm_valid, pwv_valid

ptt_val, chest_feet, arm_feet, pwv_val = compute_ptt_pwv(
    feet_chest, feet_arm, fs, DISTANCE_M,
    ptt_min_ms=PTT_MIN_MS, ptt_max_ms=PTT_MAX_MS)

if len(ptt_val) > 0:
    print(f"\nРезультаты:")
    print(f"  Валидных пар: {len(ptt_val)}")
    print(f"  PTT (медиана): {np.median(ptt_val)*1000:.1f} мс")
    print(f"  PWV (медиана): {np.median(pwv_val):.2f} м/с")
else:
    print("Валидных PTT не найдено. Проверьте окна детекции.")

# ═══════════════════════════════════════════════════════
# 6. ИНТЕРАКТИВНАЯ ВИЗУАЛИЗАЦИЯ
# ═══════════════════════════════════════════════════════
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    subplot_titles=('ЭКГ + R‑пики', 'Грудь (lowpass) + foot', 'Рука (lowpass) + foot'),
    vertical_spacing=0.07)

fig.add_trace(go.Scattergl(x=time, y=ecg_filt,
                           name='ЭКГ', line=dict(color='blue', width=0.7)), row=1, col=1)
fig.add_trace(go.Scattergl(x=time[rpeaks], y=ecg_filt[rpeaks],
                           mode='markers', name='R‑пики',
                           marker=dict(color='red', size=6, symbol='x')), row=1, col=1)

fig.add_trace(go.Scattergl(x=time, y=ch_chest_lp,
                           name='Грудь', line=dict(color='green', width=0.7)), row=2, col=1)
fig.add_trace(go.Scattergl(x=time[feet_chest], y=ch_chest_lp[feet_chest],
                           mode='markers', name='Foot грудь',
                           marker=dict(color='darkgreen', size=8, symbol='circle-open')), row=2, col=1)

fig.add_trace(go.Scattergl(x=time, y=ch_wrist_lp,
                           name='Рука', line=dict(color='purple', width=0.7)), row=3, col=1)
fig.add_trace(go.Scattergl(x=time[feet_arm], y=ch_wrist_lp[feet_arm],
                           mode='markers', name='Foot рука',
                           marker=dict(color='darkviolet', size=8, symbol='circle-open')), row=3, col=1)

if len(ptt_val) > 0:
    for i in range(min(5, len(ptt_val))):
        fc = chest_feet[i]
        fa = arm_feet[i]
        fig.add_trace(go.Scattergl(
            x=[time[fc], time[fa]],
            y=[ch_chest_lp[fc], ch_wrist_lp[fa]],
            mode='lines+markers',
            line=dict(color='black', dash='dot', width=1),
            marker=dict(size=5, color='black'),
            showlegend=(i==0),
            name=f'PTT={ptt_val[i]*1000:.0f} мс'), row=2, col=1)

fig.update_xaxes(rangeslider=dict(visible=True), row=3, col=1)
fig.update_xaxes(title_text='Время, с', row=3, col=1)
fig.update_yaxes(title_text='Z‑score', row=1, col=1)
fig.update_yaxes(title_text='Ампл.', row=2, col=1)
fig.update_yaxes(title_text='Ампл.', row=3, col=1)
fig.update_layout(height=800, title='Детекция foot методом min‑before‑peak',
                  hovermode='x unified')
fig.show()

# ═══════════════════════════════════════════════════════
# 7. ОБУЧЕНИЕ CNN (если данных достаточно)
# ═══════════════════════════════════════════════════════
if len(ptt_val) > 10:
    print("\nФормирование датасета для ML...")
    win_before = int(0.2 * fs)
    win_after  = int(0.6 * fs)
    win_len = win_before + win_after

    X, y = [], []
    for fc, pwv in zip(chest_feet, pwv_val):
        if fc - win_before < 0 or fc + win_after >= len(ch_chest_lp):
            continue
        seg_c = ch_chest_lp[fc - win_before : fc + win_after]
        seg_a = ch_wrist_lp[fc - win_before : fc + win_after]
        seg_c = (seg_c - seg_c.min()) / (seg_c.max() - seg_c.min() + 1e-9)
        seg_a = (seg_a - seg_a.min()) / (seg_a.max() - seg_a.min() + 1e-9)
        X.append(np.column_stack([seg_c, seg_a]))
        y.append(pwv)

    X = np.array(X, dtype='float32')
    y = np.array(y, dtype='float32')
    print(f"Обучающих примеров: {len(X)}")

    from sklearn.model_selection import train_test_split
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

    model = Sequential([
        Conv1D(32, kernel_size=5, activation='relu', input_shape=(win_len, 2)),
        MaxPooling1D(2),
        Conv1D(64, kernel_size=3, activation='relu'),
        MaxPooling1D(2),
        Flatten(),
        Dropout(0.5),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    print("Обучение модели...")
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test),
                        epochs=80, batch_size=8, verbose=1)
    model.save('pwv_cnn_cleaned.h5')
    print("Модель сохранена.")

    pred = model.predict(X_test).flatten()
    mae = np.mean(np.abs(pred - y_test))
    print(f"MAE на тесте: {mae:.2f} м/с")
else:
    print("Недостаточно данных для обучения (нужно >10 циклов).")

Output hidden; open in https://colab.research.google.com to view.

In [4]:
# -*- coding: utf-8 -*-
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks, savgol_filter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ─── НАСТРОЙКИ ───────────────────────────────────────
DATA_FILE   = 'data4ch_1_2.csv'      # ваш файл
DISTANCE_M  = 0.5                  # расстояние грудь–запястье, м
BEST_TKEO   = 0.6
MIN_RR_SEC  = 0.4

FOOT_START_MS = 50                   # начало поиска foot после R-пика
FOOT_END_MS   = 400

PTT_MIN_MS = 20                      # допустимые пределы PTT
PTT_MAX_MS = 200

# ═════════════════════════════════════════════════════
# 1. ЗАГРУЗКА
# ═════════════════════════════════════════════════════
print("Загрузка данных...")
df = pd.read_csv(DATA_FILE, header=None)
time = df.iloc[:, 0].values.astype(float)
time = time / 488 * 60
ch_chest_raw = df.iloc[:, 1].values.astype(float)
ch_wrist_raw = df.iloc[:, 2].values.astype(float)
ecg_raw = df.iloc[:, 4].values.astype(float)

fs = 1.0 / np.median(np.diff(time))
print(f"Частота дискретизации: {fs:.2f} Гц")

# ═════════════════════════════════════════════════════
# 2. ДЕТЕКЦИЯ R-ПИКОВ (ваш метод, без изменений)
# ═════════════════════════════════════════════════════
def butter_bandpass(data, low, high, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, min(high/nyq, 0.99)], btype='band')
    return filtfilt(b, a, data)

def preprocess_ecg_safe(raw, fs):
    ecg = butter_bandpass(raw, 0.5, min(40.0, fs*0.45), fs)  # теперь можем до 40 Гц
    return (ecg - ecg.mean()) / (ecg.std() + 1e-8)

ecg_filt = preprocess_ecg_safe(ecg_raw, fs)

def detect_rpeaks_robust(ecg, fs, min_rr_sec=0.5, tkeo_factor=0.3):
    pos = np.max(ecg) - np.median(ecg)
    neg = np.median(ecg) - np.min(ecg)
    inverted = neg > pos
    ecg_proc = -ecg if inverted else ecg

    tkeo = ecg_proc[1:-1]**2 - ecg_proc[:-2] * ecg_proc[2:]
    tkeo = np.insert(tkeo, 0, 0)
    win = max(3, int(0.05 * fs) | 1)
    tkeo_smooth = savgol_filter(tkeo, window_length=win, polyorder=2)
    thresh_tkeo = tkeo_factor * np.percentile(tkeo_smooth, 98)
    min_dist = int(min_rr_sec * fs)
    candidates, _ = find_peaks(tkeo_smooth, height=thresh_tkeo, distance=min_dist)

    half_win = max(1, int(0.03 * fs))
    peak_indices = []
    peak_heights = []
    for p in candidates:
        lo = max(0, p - half_win)
        hi = min(len(ecg), p + half_win + 1)
        local_peak = lo + (np.argmin(ecg[lo:hi]) if inverted else np.argmax(ecg[lo:hi]))
        win_bl = int(1.0 * fs)
        bl_start = max(0, local_peak - win_bl)
        bl_end = min(len(ecg), local_peak + win_bl)
        baseline = np.median(ecg[bl_start:bl_end])
        height = np.abs(ecg[local_peak] - baseline)
        peak_indices.append(local_peak)
        peak_heights.append(height)

    peak_indices = np.array(peak_indices)
    peak_heights = np.array(peak_heights)
    if len(peak_heights) < 2:
        return peak_indices
    height_threshold = 0.5 * np.median(peak_heights)
    keep = peak_heights >= height_threshold
    return np.unique(peak_indices[keep])

print("Поиск R-пиков...")
rpeaks = detect_rpeaks_robust(ecg_filt, fs, min_rr_sec=MIN_RR_SEC, tkeo_factor=BEST_TKEO)
print(f"Найдено R-пиков: {len(rpeaks)}")

# ═════════════════════════════════════════════════════
# 3. ОЧИСТКА ФПГ (замена нулей + lowpass 10 Гц)
# ═════════════════════════════════════════════════════
def remove_zeros_forward_fill(sig):
    sig = sig.copy()
    for i in range(1, len(sig)):
        if sig[i] == 0:
            sig[i] = (sig[i-1]+sig[i+1])/2
    if sig[0] == 0:
        first_nonzero = np.argmax(sig != 0)
        sig[0] = sig[first_nonzero] if first_nonzero < len(sig) else 0
    if sig[-1] == 0:
        sig[-1] = sig[-2]
    return sig

def lowpass_filter(data, fs, cutoff=40, order=3):
    nyq = 0.5 * fs #Частота Найквиста?
    #Частота, передаваемая в фильтр, не должна превышать частоту Найквиста
    b, a = butter(order, cutoff / nyq, btype='low') #order - порядок фильтра, cutoff - частота среза
    return filtfilt(b, a, data)

print("Очистка сигналов...")
ch_chest_clean = remove_zeros_forward_fill(ch_chest_raw)
ch_wrist_clean = remove_zeros_forward_fill(ch_wrist_raw)
ch_chest_lp = lowpass_filter(ch_chest_clean, fs, cutoff=10.0)
ch_wrist_lp = lowpass_filter(ch_wrist_clean, fs, cutoff=10.0)

# ═════════════════════════════════════════════════════
# 4. ДЕТЕКЦИЯ FOOT (min‑before‑peak)
# ═════════════════════════════════════════════════════
def find_foot_in_window(signal, start_idx, end_idx):
    if start_idx < 0 or end_idx > len(signal) or end_idx <= start_idx:
        return None
    seg = signal[start_idx:end_idx]
    return start_idx + np.argmin(seg)

def detect_foot_min_before_peak(signal, rpeaks, fs,
                                search_start_ms=50, search_end_ms=400):
    start_dt = int(search_start_ms * fs / 1000)
    end_dt   = int(search_end_ms * fs / 1000)
    feet = []
    for r in rpeaks:
        lo = r + start_dt
        hi = r + end_dt
        if hi >= len(signal):
            continue
        seg = signal[lo:hi]
        if len(seg) < 3:
            continue
        peak_rel = np.argmax(seg)                     # систолический пик
        search_end = max(1, int(peak_rel * 0.8))      # не дальше 80% до пика
        foot_rel = np.argmin(seg[:search_end])
        feet.append(lo + foot_rel)
    return np.array(feet, dtype=int)

print("Поиск foot на груди...")
feet_chest = detect_foot_min_before_peak(ch_chest_lp, rpeaks, fs,
                                         FOOT_START_MS, FOOT_END_MS)
print(f"Найдено foot груди: {len(feet_chest)}")

# ═════════════════════════════════════════════════════
# 5. ПОИСК FOOT НА РУКЕ С УЧЁТОМ ЗАДЕРЖКИ
# ═════════════════════════════════════════════════════
print("Поиск foot на руке с задержкой...")
feet_arm = []
for fc in feet_chest:
    start = fc + int(PTT_MIN_MS * fs / 1000)
    end   = fc + int(PTT_MAX_MS * fs / 1000)
    fa = find_foot_in_window(ch_wrist_lp, start, end)
    if fa is not None:
        feet_arm.append(fa)
    else:
        feet_arm.append(-1)

feet_arm = np.array(feet_arm, dtype=int)
valid = feet_arm >= 0
feet_chest_valid = feet_chest[valid]
feet_arm_valid = feet_arm[valid]
print(f"Валидных пар foot: {len(feet_chest_valid)}")

# ═════════════════════════════════════════════════════
# 6. РАСЧЁТ PTT И PWV
# ═════════════════════════════════════════════════════
ptt_arr = (feet_arm_valid - feet_chest_valid) / fs
pwv_arr = DISTANCE_M / ptt_arr

# MAD-фильтр выбросов
med_ptt = np.median(ptt_arr)
mad = np.median(np.abs(ptt_arr - med_ptt))
if mad > 0:
    z = np.abs(ptt_arr - med_ptt) / (mad * 1.4826)
    keep = z < 3.5
else:
    keep = np.ones(len(ptt_arr), dtype=bool)

ptt_val = ptt_arr[keep]
pwv_val = pwv_arr[keep]
feet_chest_final = feet_chest_valid[keep]
feet_arm_final = feet_arm_valid[keep]

if len(ptt_val) > 0:
    print(f"\nРезультаты:")
    print(f"  Валидных пар после фильтрации: {len(ptt_val)}")
    print(f"  PTT (медиана): {np.median(ptt_val)*1000:.1f} мс")
    print(f"  PWV (медиана): {np.median(pwv_val):.2f} м/с")
else:
    print("Не найдено валидных PTT.")

# ═════════════════════════════════════════════════════
# 7. ИНТЕРАКТИВНАЯ ВИЗУАЛИЗАЦИЯ
# ═════════════════════════════════════════════════════
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    subplot_titles=('ЭКГ + R‑пики', 'Грудь (lowpass 10 Гц) + foot', 'Рука (lowpass 10 Гц) + foot'),
    vertical_spacing=0.07)

fig.add_trace(go.Scattergl(x=time, y=ecg_filt,
                           name='ЭКГ', line=dict(color='blue', width=0.7)), row=1, col=1)
fig.add_trace(go.Scattergl(x=time[rpeaks], y=ecg_filt[rpeaks],
                           mode='markers', name='R‑пики',
                           marker=dict(color='red', size=6, symbol='x')), row=1, col=1)

fig.add_trace(go.Scattergl(x=time, y=ch_chest_lp,
                           name='Грудь', line=dict(color='green', width=0.7)), row=2, col=1)
fig.add_trace(go.Scattergl(x=time[feet_chest], y=ch_chest_lp[feet_chest],
                           mode='markers', name='Foot грудь',
                           marker=dict(color='darkgreen', size=8, symbol='circle-open')), row=2, col=1)

fig.add_trace(go.Scattergl(x=time, y=ch_wrist_lp,
                           name='Рука', line=dict(color='purple', width=0.7)), row=3, col=1)
fig.add_trace(go.Scattergl(x=time[feet_arm_valid], y=ch_wrist_lp[feet_arm_valid],
                           mode='markers', name='Foot рука',
                           marker=dict(color='darkviolet', size=8, symbol='circle-open')), row=3, col=1)

if len(ptt_val) > 0:
    for i in range(min(5, len(ptt_val))):
        fc = feet_chest_final[i]
        fa = feet_arm_final[i]
        fig.add_trace(go.Scattergl(
            x=[time[fc], time[fa]],
            y=[ch_chest_lp[fc], ch_wrist_lp[fa]],
            mode='lines+markers',
            line=dict(color='black', dash='dot', width=1),
            marker=dict(size=5, color='black'),
            showlegend=(i==0),
            name=f'PTT={ptt_val[i]*1000:.0f} мс'), row=2, col=1)

fig.update_xaxes(rangeslider=dict(visible=True), row=3, col=1)
fig.update_xaxes(title_text='Время, с', row=3, col=1)
fig.update_yaxes(title_text='Z‑score', row=1, col=1)
fig.update_yaxes(title_text='Ампл.', row=2, col=1)
fig.update_yaxes(title_text='Ампл.', row=3, col=1)
fig.update_layout(height=800, title='Детекция foot (fs ≈ 203 Гц)',
                  hovermode='x unified')
fig.show()

# ═════════════════════════════════════════════════════
# 8. ОБУЧЕНИЕ CNN (на окнах вокруг foot груди)
# ═════════════════════════════════════════════════════
if len(ptt_val) > 10:
    print("\nФормирование датасета для ML...")
    win_before = int(0.1 * fs)   # 100 мс до foot
    win_after  = int(0.5 * fs)   # 500 мс после
    win_len = win_before + win_after

    X, y = [], []
    for fc, pwv in zip(feet_chest_final, pwv_val):
        if fc - win_before < 0 or fc + win_after >= len(ch_chest_lp):
            continue
        seg_c = ch_chest_lp[fc - win_before : fc + win_after]
        seg_a = ch_wrist_lp[fc - win_before : fc + win_after]
        # min‑max нормализация
        seg_c = (seg_c - seg_c.min()) / (seg_c.max() - seg_c.min() + 1e-9)
        seg_a = (seg_a - seg_a.min()) / (seg_a.max() - seg_a.min() + 1e-9)
        X.append(np.column_stack([seg_c, seg_a]))
        y.append(pwv)

    X = np.array(X, dtype='float32')
    y = np.array(y, dtype='float32')
    print(f"Обучающих примеров: {len(X)}")

    from sklearn.model_selection import train_test_split
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

    model = Sequential([
        Conv1D(32, kernel_size=7, activation='relu', input_shape=(win_len, 2)),
        MaxPooling1D(2),
        Conv1D(64, kernel_size=5, activation='relu'),
        MaxPooling1D(2),
        Conv1D(128, kernel_size=3, activation='relu'),
        MaxPooling1D(2),
        Flatten(),
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    print("Обучение модели...")
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test),
                        epochs=80, batch_size=16, verbose=1)
    model.save('pwv_cnn_203hz.h5')
    print("Модель сохранена.")

    pred = model.predict(X_test).flatten()
    mae = np.mean(np.abs(pred - y_test))
    print(f"MAE на тесте: {mae:.2f} м/с")
else:
    print("Недостаточно данных для обучения (нужно >10 циклов).")

Output hidden; open in https://colab.research.google.com to view.



---



---



---



---



In [ ]:
# -*- coding: utf-8 -*-
"""
Пайплайн пакетной обработки ЭКГ + ФПГ → PWV
Читает config.json из папки с данными на Google Drive,
обрабатывает файлы пациентов, показывает графики,
формирует датасет и обучает CNN.
"""
import os, json
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks, savgol_filter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Монтирование Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ═══════════ ПУТИ И КОНФИГУРАЦИЯ ═══════════
DATA_ROOT = '/content/drive/MyDrive/Colab Notebooks/WORK/DATA'   # <-- ваша корневая папка
CONFIG_PATH = os.path.join(DATA_ROOT, 'config.json')

with open(CONFIG_PATH, 'r') as f:
    cfg = json.load(f)

patients_cfg = cfg['patients']
global_cfg = cfg['global']
output_dir = os.path.join(DATA_ROOT, global_cfg['output_dir'])
os.makedirs(output_dir, exist_ok=True)

# ═══════════ ФУНКЦИИ ОБРАБОТКИ СИГНАЛОВ ═══════════
def butter_bandpass(data, low, high, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [low / nyq, min(high / nyq, 0.99)], btype='band')
    return filtfilt(b, a, data)

def lowpass_filter(data, fs, cutoff=40, order=3):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype='low')
    return filtfilt(b, a, data)

def remove_zeros_forward_fill(sig):
    sig = sig.copy()
    for i in range(1, len(sig)-1):
        if sig[i] == 0:
            sig[i] = (sig[i-1] + sig[i+1]) / 2
    if sig[0] == 0:
        first_nonzero = np.argmax(sig != 0)
        sig[0] = sig[first_nonzero] if first_nonzero < len(sig) else 0
    if sig[-1] == 0:
        sig[-1] = sig[-2]
    return sig

def preprocess_ecg(raw, fs, low=0.5, high=40.0):
    ecg = butter_bandpass(raw, low, min(high, fs*0.45), fs)
    return (ecg - ecg.mean()) / (ecg.std() + 1e-9)

def detect_rpeaks_robust(ecg, fs, min_rr_sec=0.4, tkeo_factor=0.8):
    pos = np.max(ecg) - np.median(ecg)
    neg = np.median(ecg) - np.min(ecg)
    inverted = neg > pos
    ecg_proc = -ecg if inverted else ecg

    tkeo = ecg_proc[1:-1]**2 - ecg_proc[:-2] * ecg_proc[2:]
    tkeo = np.insert(tkeo, 0, 0)
    win = max(3, int(0.05 * fs) | 1)
    tkeo_smooth = savgol_filter(tkeo, window_length=win, polyorder=2)
    thresh = tkeo_factor * np.percentile(tkeo_smooth, 98)
    min_dist = int(min_rr_sec * fs)
    candidates, _ = find_peaks(tkeo_smooth, height=thresh, distance=min_dist)

    half_win = max(1, int(0.03 * fs))
    peaks, heights = [], []
    for p in candidates:
        lo, hi = max(0, p - half_win), min(len(ecg), p + half_win + 1)
        local_peak = lo + (np.argmin(ecg[lo:hi]) if inverted else np.argmax(ecg[lo:hi]))
        bl_start = max(0, local_peak - int(1.0 * fs))
        bl_end = min(len(ecg), local_peak + int(1.0 * fs))
        baseline = np.median(ecg[bl_start:bl_end])
        heights.append(abs(ecg[local_peak] - baseline))
        peaks.append(local_peak)
    peaks, heights = np.array(peaks), np.array(heights)
    if len(heights) < 2:
        return peaks
    keep = heights >= 0.5 * np.median(heights)
    return np.unique(peaks[keep])

def find_foot_in_window(signal, start_idx, end_idx):
    if start_idx < 0 or end_idx > len(signal) or end_idx <= start_idx:
        return None
    return start_idx + np.argmin(signal[start_idx:end_idx])

def detect_foot_min_before_peak(signal, rpeaks, fs,
                                search_start_ms=50, search_end_ms=400):
    start_dt = int(search_start_ms * fs / 1000)
    end_dt   = int(search_end_ms * fs / 1000)
    feet = []
    for r in rpeaks:
        lo = r + start_dt
        hi = r + end_dt
        if hi >= len(signal):
            continue
        seg = signal[lo:hi]
        if len(seg) < 3:
            continue
        peak_rel = np.argmax(seg)
        search_end = max(1, int(peak_rel * 0.8))
        foot_rel = np.argmin(seg[:search_end])
        feet.append(lo + foot_rel)
    return np.array(feet, dtype=int)

# ═══════════ ОБРАБОТКА ОДНОГО ФАЙЛА ═══════════
def process_one_file(file_path, dist_m, tkeo_f, ppg_cut):
    """Возвращает словарь с ключами:
    'X', 'y', 'stats', 'fig' (объект plotly) или None при ошибке.
    """
    df = pd.read_csv(file_path, header=None)
    time = df.iloc[:, 0].values.astype(float)
    # Преобразование времени: ваш способ
    time = time / 488 * 60
    ch_chest_raw = df.iloc[:, 1].values.astype(float)
    ch_wrist_raw = df.iloc[:, 2].values.astype(float)
    ecg_raw      = df.iloc[:, 4].values.astype(float)

    fs = 1.0 / np.median(np.diff(time))
    if fs < 20 or fs > 500:
        print("    Некорректная частота дискретизации")
        return None

    # ЭКГ
    ecg_filt = preprocess_ecg(ecg_raw, fs,
                              low=global_cfg['ecg_bandpass_low'],
                              high=global_cfg['ecg_bandpass_high'])
    rpeaks = detect_rpeaks_robust(ecg_filt, fs,
                                  min_rr_sec=global_cfg['r_peak_min_rr_sec'],
                                  tkeo_factor=tkeo_f)
    if len(rpeaks) < 3:
        print("    Недостаточно R-пиков")
        return None

    # Очистка ФПГ
    ch_chest_clean = remove_zeros_forward_fill(ch_chest_raw)
    ch_wrist_clean = remove_zeros_forward_fill(ch_wrist_raw)
    ch_chest_lp = lowpass_filter(ch_chest_clean, fs, cutoff=ppg_cut, order=3)
    ch_wrist_lp = lowpass_filter(ch_wrist_clean, fs, cutoff=ppg_cut, order=3)

    # Foot груди
    feet_chest = detect_foot_min_before_peak(
        ch_chest_lp, rpeaks, fs,
        global_cfg['foot_search_start_ms'],
        global_cfg['foot_search_end_ms'])
    if len(feet_chest) < 3:
        print("    Мало foot на груди")
        return None

    # Foot руки с учётом задержки
    ptt_min_dt = int(global_cfg['ptt_min_ms'] * fs / 1000)
    ptt_max_dt = int(global_cfg['ptt_max_ms'] * fs / 1000)
    feet_arm = []
    for fc in feet_chest:
        fa = find_foot_in_window(ch_wrist_lp, fc + ptt_min_dt, fc + ptt_max_dt)
        feet_arm.append(fa if fa is not None else -1)
    feet_arm = np.array(feet_arm, dtype=int)
    valid = feet_arm >= 0
    if valid.sum() < 3:
        print("    Мало валидных пар foot")
        return None

    ptt_arr = (feet_arm[valid] - feet_chest[valid]) / fs
    pwv_arr = dist_m / ptt_arr

    # MAD-фильтр выбросов
    med = np.median(ptt_arr)
    mad = np.median(np.abs(ptt_arr - med))
    if mad > 0:
        keep = np.abs(ptt_arr - med) / (mad * 1.4826) < 3.5
    else:
        keep = np.ones(len(ptt_arr), dtype=bool)
    ptt_val = ptt_arr[keep]
    pwv_val = pwv_arr[keep]
    chest_fin = feet_chest[valid][keep]
    arm_fin   = feet_arm[valid][keep]

    if len(ptt_val) < 5:
        print("    Недостаточно циклов после фильтрации")
        return None

    # Окна для CNN
    wb = int(global_cfg['model_input_win_before_ms'] * fs / 1000)
    wa = int(global_cfg['model_input_win_after_ms'] * fs / 1000)
    X_list, y_list = [], []
    for fc, pwv in zip(chest_fin, pwv_val):
        if fc - wb < 0 or fc + wa >= len(ch_chest_lp):
            continue
        seg_c = ch_chest_lp[fc - wb : fc + wa]
        seg_a = ch_wrist_lp[fc - wb : fc + wa]
        seg_c = (seg_c - seg_c.min()) / (seg_c.max() - seg_c.min() + 1e-9)
        seg_a = (seg_a - seg_a.min()) / (seg_a.max() - seg_a.min() + 1e-9)
        X_list.append(np.column_stack([seg_c, seg_a]))
        y_list.append(pwv)
    X = np.array(X_list, dtype='float32')
    y = np.array(y_list, dtype='float32')

    # Визуализация
    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True,
        subplot_titles=('ЭКГ + R‑пики', 'Грудь (lowpass) + foot', 'Рука (lowpass) + foot'),
        vertical_spacing=0.07)
    fig.add_trace(go.Scattergl(x=time, y=ecg_filt,
                               name='ЭКГ', line=dict(color='blue', width=0.7)), row=1, col=1)
    fig.add_trace(go.Scattergl(x=time[rpeaks], y=ecg_filt[rpeaks],
                               mode='markers', name='R‑пики',
                               marker=dict(color='red', size=5, symbol='x')), row=1, col=1)
    fig.add_trace(go.Scattergl(x=time, y=ch_chest_lp,
                               name='Грудь', line=dict(color='green', width=0.7)), row=2, col=1)
    fig.add_trace(go.Scattergl(x=time[feet_chest], y=ch_chest_lp[feet_chest],
                               mode='markers', name='Foot грудь',
                               marker=dict(color='darkgreen', size=6, symbol='circle-open')), row=2, col=1)
    fig.add_trace(go.Scattergl(x=time, y=ch_wrist_lp,
                               name='Рука', line=dict(color='purple', width=0.7)), row=3, col=1)
    fig.add_trace(go.Scattergl(x=time[feet_arm[valid]], y=ch_wrist_lp[feet_arm[valid]],
                               mode='markers', name='Foot рука',
                               marker=dict(color='darkviolet', size=6, symbol='circle-open')), row=3, col=1)
    for i in range(min(5, len(ptt_val))):
        fc, fa = chest_fin[i], arm_fin[i]
        fig.add_trace(go.Scattergl(
            x=[time[fc], time[fa]],
            y=[ch_chest_lp[fc], ch_wrist_lp[fa]],
            mode='lines+markers',
            line=dict(color='black', dash='dot', width=1),
            marker=dict(size=4, color='black'),
            showlegend=(i == 0),
            name=f'PTT={ptt_val[i]*1000:.0f} мс'), row=2, col=1)
    fig.update_xaxes(rangeslider=dict(visible=True), row=3, col=1)
    fig.update_xaxes(title_text='Время, с', row=3, col=1)
    fig.update_yaxes(title_text='Z‑score', row=1, col=1)
    fig.update_yaxes(title_text='Ампл.', row=2, col=1)
    fig.update_yaxes(title_text='Ампл.', row=3, col=1)
    fig.update_layout(height=800, title=os.path.basename(file_path),
                      hovermode='x unified')

    return {
        'X': X, 'y': y,
        'stats': {
            'file': os.path.basename(file_path),
            'n_cycles': len(ptt_val),
            'median_ptt_ms': np.median(ptt_val)*1000,
            'median_pwv': np.median(pwv_val)
        },
        'fig': fig
    }





In [ ]:
# ═══════════ ГЛАВНЫЙ ЦИКЛ ПО ПАЦИЕНТАМ И ФАЙЛАМ ═══════════
X_all, y_all = [], []
stats_all = []

for patient in patients_cfg:
    print(f"\n{'='*60}\nПациент {patient['id']}\n{'='*60}")
    for fname in patient['files']:
        full_path = os.path.join(DATA_ROOT, fname)
        if not os.path.exists(full_path):
            print(f"  Файл {fname} не найден, пропуск")
            continue
        print(f"  Обработка {fname}...")
        res = process_one_file(
            full_path,
            dist_m=patient['distance_m'],
            tkeo_f=patient['tkeo_factor'],
            ppg_cut=patient['ppg_cutoff_hz']
        )
        if res is None:
            print("    Ошибка или недостаточно циклов, пропуск")
            continue
        # Показываем интерактивный график
        res['fig'].show()
        ans = input("    Добавить данные в обучающую выборку? [y/n]: ").strip().lower()
        if ans == 'y':
            X_all.append(res['X'])
            y_all.append(res['y'])
            stats_all.append(res['stats'])
            print("    ✅ Добавлено")
        else:
            print("    ❌ Пропущено")

# ═══════════ СОХРАНЕНИЕ ДАТАСЕТА ═══════════
if X_all:
    X_final = np.concatenate(X_all, axis=0)
    y_final = np.concatenate(y_all, axis=0)
    np.save(os.path.join(output_dir, 'X_all.npy'), X_final)
    np.save(os.path.join(output_dir, 'y_all.npy'), y_final)
    pd.DataFrame(stats_all).to_csv(os.path.join(output_dir, 'processing_stats.csv'), index=False)
    print(f"\nСохранено {X_final.shape[0]} окон в {output_dir}")
else:
    print("\nНет данных для сохранения. Проверьте конфигурацию и файлы.")
    exit()

In [ ]:
# ═══════════ ОБУЧЕНИЕ CNN ═══════════
print("\nОбучение CNN...")
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_final, test_size=0.2, random_state=42)

_, win_len, n_ch = X_train.shape
model = Sequential([
    Conv1D(32, kernel_size=7, activation='relu', input_shape=(win_len, n_ch)),
    MaxPooling1D(2),
    Conv1D(64, kernel_size=5, activation='relu'),
    MaxPooling1D(2),
    Conv1D(128, kernel_size=3, activation='relu'),
    MaxPooling1D(2),
    Flatten(),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = model.fit(X_train, y_train, validation_data=(X_test, y_test),
                    epochs=80, batch_size=16, verbose=1)
model.save(os.path.join(output_dir, 'pwv_cnn_model.h5'))
print("Модель сохранена.")

pred = model.predict(X_test).flatten()
mae = np.mean(np.abs(pred - y_test))
print(f"MAE на тестовой выборке: {mae:.2f} м/с")

Пайплайн для обкатки аугментации. ПРОВЕРИТЬ ВОЗМРЖНОСТЬ ИНТЕГРАЦИИ В ОСНОВНОЙ

In [ ]:
# -*- coding: utf-8 -*-
"""
Пайплайн аугментации окон здорового пациента.
Загружает X_all.npy, y_all.npy, балансирует выборку и визуализирует результат.
"""
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

# ═══════════ 1. ЗАГРУЗКА ДАННЫХ ═══════════
# Предполагаем, что X_all.npy и y_all.npy уже сохранены
X_all = np.load('X_all.npy')   # shape (N, win_len, 2)
y_all = np.load('y_all.npy')

# Определяем порог "здоровой" PWV (например, > 6 м/с)
HEALTHY_THRESHOLD = 6.0   # м/с, можно изменить
healthy_mask = y_all > HEALTHY_THRESHOLD
X_healthy = X_all[healthy_mask]
y_healthy = y_all[healthy_mask]
X_abnormal = X_all[~healthy_mask]
y_abnormal = y_all[~healthy_mask]

print(f"Всего примеров: {len(X_all)}")
print(f"Здоровых (PWV > {HEALTHY_THRESHOLD}): {len(X_healthy)}")
print(f"Аномальных: {len(X_abnormal)}")

# ═══════════ 2. ФУНКЦИИ АУГМЕНТАЦИИ ═══════════
def time_shift(window, max_shift=3):
    """Циклический сдвиг по времени"""
    shift = np.random.randint(-max_shift, max_shift + 1)
    return np.roll(window, shift, axis=0)

def amplitude_scale(window, range=(0.9, 1.1)):
    """Масштабирование амплитуды"""
    scale = np.random.uniform(*range)
    return window * scale

def add_gaussian_noise(window, std=0.02):
    """Добавление гауссовского шума"""
    noise = np.random.normal(0, std, window.shape)
    return window + noise

def time_warp(window, max_stretch=0.05):
    """Небольшое искажение временной шкалы с интерполяцией"""
    old_len = window.shape[0]
    new_len = int(old_len * (1 + np.random.uniform(-max_stretch, max_stretch)))
    idx_old = np.linspace(0, old_len - 1, old_len)
    idx_new = np.linspace(0, old_len - 1, new_len)
    warped = np.zeros((new_len, window.shape[1]))
    for ch in range(window.shape[1]):
        f = interp1d(idx_old, window[:, ch], kind='linear', fill_value='extrapolate')
        warped[:, ch] = f(idx_new)
    # Приводим к исходной длине
    if new_len > old_len:
        warped = warped[:old_len, :]
    elif new_len < old_len:
        pad = old_len - new_len
        warped = np.pad(warped, ((0, pad), (0, 0)), mode='edge')
    return warped

# ═══════════ 3. ГЕНЕРАЦИЯ НОВЫХ ПРИМЕРОВ ═══════════
def augment_healthy(X, y, multiplier=3):
    """Создаёт multiplier аугментированных копий для каждого здорового окна."""
    aug_X, aug_y = [], []
    for x, label in zip(X, y):
        aug_X.append(x)
        aug_y.append(label)
        for _ in range(multiplier):
            x_new = x.copy()
            # Применяем случайные комбинации преобразований
            if np.random.rand() > 0.5:
                x_new = time_shift(x_new)
            if np.random.rand() > 0.5:
                x_new = amplitude_scale(x_new)
            if np.random.rand() > 0.5:
                x_new = add_gaussian_noise(x_new, std=0.015)
            if np.random.rand() > 0.3:   # реже применяем time_warp
                x_new = time_warp(x_new)
            aug_X.append(x_new)
            aug_y.append(label)
    return np.array(aug_X), np.array(aug_y)

print("Аугментация здоровых примеров...")
X_healthy_aug, y_healthy_aug = augment_healthy(X_healthy, y_healthy, multiplier=3)
print(f"После аугментации здоровых: {len(X_healthy_aug)} (было {len(X_healthy)})")

# ═══════════ 4. ВИЗУАЛИЗАЦИЯ (примеры) ═══════════
def plot_augmentation_samples(original_X, aug_X, n_samples=3):
    """
    Показывает n_samples случайных исходных окон и их аугментированные версии.
    """
    n = min(n_samples, len(original_X))
    indices = np.random.choice(len(original_X), n, replace=False)
    for idx in indices:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        # Оригинальное окно
        axes[0].plot(original_X[idx, :, 0], label='Грудь (исх.)')
        axes[0].plot(original_X[idx, :, 1], label='Рука (исх.)')
        axes[0].set_title('Оригинальное окно')
        axes[0].legend()
        # Несколько аугментированных копий (например, первые 3 из сгенерированных для этого индекса)
        # Так как порядок нарушен, ищем соответствующие сгенерированные окна
        # Проще: заново сгенерируем для конкретного примера
        x_orig = original_X[idx]
        axes[1].plot(x_orig[:, 0], label='Грудь (исх.)', alpha=0.5)
        axes[1].plot(x_orig[:, 1], label='Рука (исх.)', alpha=0.5)
        for i in range(3):
            x_aug = x_orig.copy()
            if np.random.rand() > 0.5: x_aug = time_shift(x_aug)
            if np.random.rand() > 0.5: x_aug = amplitude_scale(x_aug)
            if np.random.rand() > 0.5: x_aug = add_gaussian_noise(x_aug, std=0.015)
            if np.random.rand() > 0.3: x_aug = time_warp(x_aug)
            axes[1].plot(x_aug[:, 0], linestyle='--', label=f'Грудь ауг.{i+1}')
            axes[1].plot(x_aug[:, 1], linestyle='--', label=f'Рука ауг.{i+1}')
        axes[1].set_title('Аугментированные варианты')
        axes[1].legend()
        plt.suptitle(f'Пример окна здорового пациента (PWV={y_healthy[idx]:.2f} м/с)')
        plt.tight_layout()
        plt.show()

plot_augmentation_samples(X_healthy, X_healthy_aug, n_samples=2)

# ═══════════ 5. ОБУЧЕНИЕ МОДЕЛИ ДО И ПОСЛЕ АУГМЕНТАЦИИ ═══════════
def build_cnn(input_shape):
    model = Sequential([
        Conv1D(32, kernel_size=7, activation='relu', input_shape=input_shape),
        MaxPooling1D(2),
        Conv1D(64, kernel_size=5, activation='relu'),
        MaxPooling1D(2),
        Conv1D(128, kernel_size=3, activation='relu'),
        MaxPooling1D(2),
        Flatten(),
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Объединение с аномальными примерами (без изменений)
X_balanced = np.concatenate([X_healthy_aug, X_abnormal], axis=0)
y_balanced = np.concatenate([y_healthy_aug, y_abnormal], axis=0)

# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced, test_size=0.2, random_state=42)
print(f"Обучающая выборка: {X_train.shape[0]} примеров")

# Модель с аугментацией
model_aug = build_cnn((X_train.shape[1], 2))
print("Обучение с аугментацией...")
hist_aug = model_aug.fit(X_train, y_train, validation_data=(X_test, y_test),
                          epochs=60, batch_size=16, verbose=0)
pred_aug = model_aug.predict(X_test).flatten()
mae_aug = np.mean(np.abs(pred_aug - y_test))
print(f"MAE с аугментацией: {mae_aug:.2f} м/с")

# Модель БЕЗ аугментации (на исходных данных)
X_orig = np.concatenate([X_healthy, X_abnormal], axis=0)
y_orig = np.concatenate([y_healthy, y_abnormal], axis=0)
X_tr_o, X_te_o, y_tr_o, y_te_o = train_test_split(X_orig, y_orig, test_size=0.2, random_state=42)

model_noaug = build_cnn((X_tr_o.shape[1], 2))
print("Обучение без аугментации...")
hist_noaug = model_noaug.fit(X_tr_o, y_tr_o, validation_data=(X_te_o, y_te_o),
                             epochs=60, batch_size=16, verbose=0)
pred_noaug = model_noaug.predict(X_te_o).flatten()
mae_noaug = np.mean(np.abs(pred_noaug - y_te_o))
print(f"MAE без аугментации: {mae_noaug:.2f} м/с")

print("\nСравнение MAE:")
print(f"  Без аугментации: {mae_noaug:.2f} м/с")
print(f"  С аугментацией:  {mae_aug:.2f} м/с")